In [0]:
import os
print(os.getcwd())
print(os.listdir())

In [0]:
import os

for root, dirs, files in os.walk("."):
    level = root.count(os.sep)
    if level <= 2:
        print(root)

In [0]:
import sys
import os

print("Python:", sys.version)
print("Working directory:", os.getcwd())

from factorymind.simulation.machine import MachineSimulator

print("FactoryMind import: OK")
print("MachineSimulator: OK")

In [0]:
import inspect
from factorymind.simulation.machine import MachineSimulator

print(inspect.signature(MachineSimulator))
print()
print(inspect.signature(MachineSimulator.run))
print()
print(inspect.signature(MachineSimulator.step))

In [0]:
from pathlib import Path

generator = Path("scripts/generate_dataset.py")

print(generator.exists())

print(generator.read_text()[:12000])

In [0]:
import importlib.util

packages = [
    "numpy",
    "pandas",
    "scipy",
    "sklearn",
    "mlflow",
    "pyspark",
]

for package in packages:
    print(f"{package:12}:", "INSTALLED" if importlib.util.find_spec(package) else "MISSING")

In [0]:
from factorymind.domain.models import TelemetrySnapshot
import inspect

print(inspect.signature(TelemetrySnapshot))
print()
print(TelemetrySnapshot.__annotations__)

In [0]:
from factorymind.simulation.machine import MachineSimulator
from factorymind.domain.models import MachineConfig

sim = MachineSimulator(
    config=MachineConfig(machine_id="TEST-001"),
    seed=42
)

snapshots = sim.run(10)

print("Number of snapshots:", len(snapshots))
print()
print(snapshots[0])
print()
print("Snapshot attributes:")
print(vars(snapshots[0]))
print("Catalogs:")
display(spark.sql("SHOW CATALOGS"))
print("Current database:")
display(spark.sql("SELECT current_catalog(), current_schema()"))
display(spark.sql("SHOW SCHEMAS"))
display(spark.sql("SELECT current_catalog(), current_schema()"))
display(spark.sql("SHOW SCHEMAS"))

In [0]:
# Show Databricks catalog/schema information as plain text

catalogs = spark.sql("SHOW CATALOGS").collect()

print("=== CATALOGS ===")
for row in catalogs:
    print(row[0])

print("\n=== CURRENT CATALOG / SCHEMA ===")
current = spark.sql(
    "SELECT current_catalog(), current_schema()"
).collect()[0]

print("Catalog:", current[0])
print("Schema :", current[1])

print("\n=== SCHEMAS ===")
schemas = spark.sql("SHOW SCHEMAS").collect()

for row in schemas:
    print(row[0])
    print("=== TABLES IN CURRENT SCHEMA ===")

tables = spark.sql("SHOW TABLES").collect()

for row in tables:
    print(
        f"database={row.database} | "
        f"table={row.tableName} | "
        f"temporary={row.isTemporary}"
    )

In [0]:
# ============================================================
# FACTORYMIND — ML DATASET V1
# Generate 100 machines × 1000 timesteps
# ============================================================

from factorymind.domain.enums import FailureMode
from factorymind.domain.models import MachineConfig
from factorymind.simulation.machine import MachineSimulator

import pandas as pd
import numpy as np

# Configuration
NUM_MACHINES = 100
DURATION = 1000
SEED = 42
FAILURE_FRACTION = 0.30

SENSOR_NAMES = [
    "temperature",
    "vibration",
    "pressure",
    "motor_current",
    "rpm",
    "load",
    "coolant_temperature",
    "coolant_flow",
    "acoustic_level",
]

FAILURE_MODES = [
    FailureMode.BEARING_DEGRADATION,
    FailureMode.BEARING_FAILURE,
    FailureMode.THERMAL_OVERLOAD,
    FailureMode.MECHANICAL_IMBALANCE,
    FailureMode.MOTOR_OVERLOAD,
    FailureMode.COOLING_FAILURE,
]

rng = np.random.default_rng(SEED)

telemetry_rows = []
machine_label_rows = []

failure_count = int(NUM_MACHINES * FAILURE_FRACTION)

failure_machine_indices = set(
    rng.choice(
        np.arange(NUM_MACHINES),
        size=failure_count,
        replace=False
    ).tolist()
)

print(f"Generating {NUM_MACHINES} machines...")
print(f"Duration per machine: {DURATION} steps")
print(f"Failure machines: {failure_count}")
print()

for i in range(NUM_MACHINES):

    machine_id = f"M-{i:04d}"

    sim = MachineSimulator(
        config=MachineConfig(machine_id=machine_id),
        seed=SEED + i,
    )

    failure_mode = FailureMode.NONE
    onset_step = None
    ramp_steps = None

    # --------------------------------------------------------
    # Inject failure for selected machines
    # --------------------------------------------------------

    if i in failure_machine_indices:
        failure_mode = FAILURE_MODES[
           int(rng.integers(0, len(FAILURE_MODES)))
        ]

        onset_step = int(
            rng.integers(
                DURATION // 4,
                DURATION // 2
            )
        )

        ramp_steps = int(
            rng.integers(100, 300)
        )

        sim.inject_failure(
            mode=failure_mode,
            onset_step=onset_step,
            ramp_steps=ramp_steps,
        )

    machine_label_rows.append({
        "machine_id": machine_id,
        "failure_mode": failure_mode.value,
        "onset_step": onset_step,
        "ramp_steps": ramp_steps,
    })

    # --------------------------------------------------------
    # Step manually.
    #
    # IMPORTANT:
    # sim.runtime now represents THIS timestep.
    # --------------------------------------------------------

    for step in range(DURATION):

        snapshot = sim.step()
        runtime = sim.runtime

        row = {
            "machine_id": machine_id,
            "step": step,
            "timestamp": snapshot.timestamp,
        }

        # Sensor values
        for sensor in SENSOR_NAMES:
            row[sensor] = snapshot.readings.get(sensor, np.nan)

        # Sensor quality
        for sensor in SENSOR_NAMES:
            row[f"{sensor}_quality"] = snapshot.quality.get(
                sensor, np.nan
            )

        # Ground truth
        row["degradation_score"] = runtime.degradation_score
        row["health_state"] = runtime.health.value

        # Anomaly ground truth
        row["is_anomalous"] = int(
            runtime.degradation_score > 0.2
        )

        # Failure information
        row["failure_mode"] = failure_mode.value
        row["onset_step"] = onset_step

        # Remaining steps until failure onset
        if onset_step is not None:
            row["steps_to_failure"] = max(
                0,
                onset_step - step
            )
        else:
            row["steps_to_failure"] = None

        telemetry_rows.append(row)

    if (i + 1) % 10 == 0:
        print(f"Generated {i + 1}/{NUM_MACHINES} machines")

# ============================================================
# Create DataFrames
# ============================================================

telemetry_pdf = pd.DataFrame(telemetry_rows)
machine_labels_pdf = pd.DataFrame(machine_label_rows)

print()
print("=" * 60)
print("DATASET GENERATION COMPLETE")
print("=" * 60)

print("Telemetry shape:", telemetry_pdf.shape)
print("Machine labels shape:", machine_labels_pdf.shape)

In [0]:
# ============================================================
# FACTORYMIND — DATASET VALIDATION
# ============================================================

print("=" * 60)
print("DATASET VALIDATION")
print("=" * 60)

print("\nShape:")
print("Telemetry:", telemetry_pdf.shape)
print("Machines :", machine_labels_pdf.shape)

print("\nFailure modes:")
print(machine_labels_pdf["failure_mode"].value_counts())

print("\nAnomaly labels:")
print(telemetry_pdf["is_anomalous"].value_counts())

print("\nHealth states:")
print(telemetry_pdf["health_state"].value_counts())

print("\nDegradation statistics:")
print(telemetry_pdf["degradation_score"].describe())

print("\nMissing values:")
print(
    telemetry_pdf.isna()
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

In [0]:
# ============================================================
# CHECK ONE FAILURE TRAJECTORY
# ============================================================

failed_machine = (
    machine_labels_pdf[
        machine_labels_pdf["failure_mode"] != "none"
    ].iloc[0]["machine_id"]
)

failure_info = machine_labels_pdf[
    machine_labels_pdf["machine_id"] == failed_machine
].iloc[0]

print("Machine:", failed_machine)
print("Failure mode:", failure_info["failure_mode"])
print("Failure onset:", failure_info["onset_step"])
print("Ramp steps:", failure_info["ramp_steps"])

trajectory = telemetry_pdf[
    telemetry_pdf["machine_id"] == failed_machine
][
    [
        "step",
        "degradation_score",
        "health_state",
        "is_anomalous",
        "failure_mode",
        "onset_step"
    ]
]

display(trajectory.iloc[::50])

In [0]:
# ============================================================
# FACTORYMIND — BRONZE DATA LAYER
# ============================================================

CATALOG = "workspace"
SCHEMA = "default"

telemetry_table = f"{CATALOG}.{SCHEMA}.factorymind_telemetry"
machines_table = f"{CATALOG}.{SCHEMA}.factorymind_machine_labels"

# Convert timestamp to Spark timestamp-compatible type
telemetry_pdf["timestamp"] = pd.to_datetime(
    telemetry_pdf["timestamp"],
    utc=True
)

telemetry_sdf = spark.createDataFrame(telemetry_pdf)
machines_sdf = spark.createDataFrame(machine_labels_pdf)

# Write Delta tables
(
    telemetry_sdf
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(telemetry_table)
)

(
    machines_sdf
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(machines_table)
)

print("Created:")
print(telemetry_table)
print(machines_table)

In [0]:
# ============================================================
# VERIFY BRONZE TABLES
# ============================================================

display(
    spark.sql(
        f"DESCRIBE TABLE {telemetry_table}"
    )
)

In [0]:
print(
    spark.sql(
        f"SELECT COUNT(*) AS rows FROM {telemetry_table}"
    ).collect()[0]["rows"]
)

print(
    spark.sql(
        f"SELECT COUNT(*) AS machines FROM {machines_table}"
    ).collect()[0]["machines"]
)

In [0]:
# ============================================================
# FACTORYMIND — SILVER FEATURE ENGINEERING
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

SOURCE = "workspace.default.factorymind_telemetry"
TARGET = "workspace.default.factorymind_features"

sensors = [
    "temperature",
    "vibration",
    "pressure",
    "motor_current",
    "rpm",
    "load",
    "coolant_temperature",
    "coolant_flow",
    "acoustic_level",
]

df = spark.table(SOURCE)

print("Input rows:", df.count())

# ------------------------------------------------------------
# Window definitions
#
# All windows are partitioned by machine and ordered by time.
# This prevents one machine's history from contaminating another.
# ------------------------------------------------------------

w_10 = (
    Window
    .partitionBy("machine_id")
    .orderBy("step")
    .rowsBetween(-9, 0)
)

w_30 = (
    Window
    .partitionBy("machine_id")
    .orderBy("step")
    .rowsBetween(-29, 0)
)

w_lag = (
    Window
    .partitionBy("machine_id")
    .orderBy("step")
)

features = df

# ------------------------------------------------------------
# Rolling features
# ------------------------------------------------------------

for sensor in sensors:

    features = (
        features
        .withColumn(
            f"{sensor}_mean_10",
            F.avg(sensor).over(w_10)
        )
        .withColumn(
            f"{sensor}_std_10",
            F.stddev(sensor).over(w_10)
        )
        .withColumn(
            f"{sensor}_mean_30",
            F.avg(sensor).over(w_30)
        )
        .withColumn(
            f"{sensor}_std_30",
            F.stddev(sensor).over(w_30)
        )
        .withColumn(
            f"{sensor}_lag_1",
            F.lag(sensor, 1).over(w_lag)
        )
        .withColumn(
            f"{sensor}_lag_5",
            F.lag(sensor, 5).over(w_lag)
        )
    )

# ------------------------------------------------------------
# Rate of change
# ------------------------------------------------------------

features = (
    features
    .withColumn(
        "temperature_delta",
        F.col("temperature") -
        F.col("temperature_lag_1")
    )
    .withColumn(
        "vibration_delta",
        F.col("vibration") -
        F.col("vibration_lag_1")
    )
    .withColumn(
        "motor_current_delta",
        F.col("motor_current") -
        F.col("motor_current_lag_1")
    )
    .withColumn(
        "coolant_flow_delta",
        F.col("coolant_flow") -
        F.col("coolant_flow_lag_1")
    )
)

# ------------------------------------------------------------
# Cross-sensor relationships
# ------------------------------------------------------------

features = (
    features
    .withColumn(
        "temperature_vibration_ratio",
        F.col("temperature") /
        (F.abs(F.col("vibration")) + F.lit(1e-6))
    )
    .withColumn(
        "vibration_current_ratio",
        F.col("vibration") /
        (F.abs(F.col("motor_current")) + F.lit(1e-6))
    )
    .withColumn(
        "load_current_ratio",
        F.col("load") /
        (F.abs(F.col("motor_current")) + F.lit(1e-6))
    )
    .withColumn(
        "coolant_flow_temperature_ratio",
        F.col("coolant_flow") /
        (F.abs(F.col("coolant_temperature")) + F.lit(1e-6))
    )
)

# ------------------------------------------------------------
# Machine-level temporal ordering
# ------------------------------------------------------------

features = features.orderBy(
    "machine_id",
    "step"
)

# ------------------------------------------------------------
# Save Silver table
# ------------------------------------------------------------

(
    features.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET)
)

print("SUCCESS")
print("Created:", TARGET)
print("Rows:", features.count())
print("Columns:", len(features.columns))

In [0]:
print(
    spark.sql(
        """
        SELECT COUNT(*) AS rows
        FROM workspace.default.factorymind_features
        """
    ).collect()[0]["rows"]
)

print(
    spark.sql(
        """
        SELECT COUNT(*) AS machines
        FROM workspace.default.factorymind_features
        """
    ).collect()[0]["machines"]
)

In [0]:
# ============================================================
# FACTORYMIND — ISOLATION FOREST ANOMALY DETECTION
# ============================================================

import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)

# ------------------------------------------------------------
# 1. Load Silver features
# ------------------------------------------------------------

df = spark.table("workspace.default.factorymind_features")

print("Input rows:", df.count())

# ------------------------------------------------------------
# 2. Identify feature columns
# ------------------------------------------------------------

GROUND_TRUTH = {
    "degradation_score",
    "health_state",
    "is_anomalous",
    "failure_mode",
    "onset_step",
    "steps_to_failure"
}

IDENTIFIERS = {
    "machine_id",
    "timestamp"
}

feature_columns = [
    c for c in df.columns
    if c not in GROUND_TRUTH
    and c not in IDENTIFIERS
]

print("Number of feature columns:", len(feature_columns))

# ------------------------------------------------------------
# 3. Remove rows with incomplete engineered features
# ------------------------------------------------------------

model_df = df.select(
    "machine_id",
    "step",
    "degradation_score",
    "is_anomalous",
    *feature_columns
)

# Drop rows containing null feature values
model_df = model_df.dropna(subset=feature_columns)

print("Rows after removing NaNs:", model_df.count())

# ------------------------------------------------------------
# 4. Convert to Pandas
# ------------------------------------------------------------

pdf = model_df.toPandas()

print("Pandas shape:", pdf.shape)

# ------------------------------------------------------------
# 5. Select training data
#
# Train ONLY on healthy observations.
# Ground truth is used ONLY to construct the clean
# training population — never as a model feature.
# ------------------------------------------------------------

healthy_train = pdf[pdf["degradation_score"] == 0].copy()

print("Healthy training rows:", len(healthy_train))

X_train = healthy_train[feature_columns]

# ------------------------------------------------------------
# 6. Train Isolation Forest
# ------------------------------------------------------------

iso_forest = IsolationForest(
    n_estimators=300,
    contamination="auto",
    max_samples="auto",
    random_state=42,
    n_jobs=-1
)

iso_forest.fit(X_train)

print("SUCCESS: Isolation Forest trained")

In [0]:
# ============================================================
# FACTORYMIND — ANOMALY SCORING
# ============================================================

# Isolation Forest:
#   higher decision_function = more normal
#   lower decision_function = more anomalous

pdf["anomaly_score"] = iso_forest.decision_function(
    pdf[feature_columns]
)

pdf["iforest_prediction"] = iso_forest.predict(
    pdf[feature_columns]
)

# sklearn:
#   +1 = normal
#   -1 = anomaly

pdf["iforest_anomaly"] = (
    pdf["iforest_prediction"] == -1
).astype(int)

print("\nPrediction distribution:")
print(pdf["iforest_anomaly"].value_counts())

print("\nAnomaly rate:")
print(pdf["iforest_anomaly"].mean())

In [0]:
# ============================================================
# FACTORYMIND — ISOLATION FOREST EVALUATION
# ============================================================

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# Ground truth
y_true = pdf["is_anomalous"].astype(int)
y_pred = pdf["iforest_anomaly"].astype(int)

# ------------------------------------------------------------
# Classification metrics
# ------------------------------------------------------------

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

# False-positive / false-negative rates
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

fpr = fp / (fp + tn)
fnr = fn / (fn + tp)

print("=" * 60)
print("ISOLATION FOREST — BASELINE RESULTS")
print("=" * 60)

print(f"Accuracy:            {accuracy:.4f}")
print(f"Precision:           {precision:.4f}")
print(f"Recall:              {recall:.4f}")
print(f"F1 Score:            {f1:.4f}")
print(f"False Positive Rate: {fpr:.4f}")
print(f"False Negative Rate: {fnr:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["Normal", "Anomaly"],
    zero_division=0
))

In [0]:
# ============================================================
# RESTORE GROUND-TRUTH COLUMNS FOR EVALUATION
# ============================================================

# Read the original Silver feature table
eval_df = spark.table("workspace.default.factorymind_features")

# Keep only evaluation columns
ground_truth_df = eval_df.select(
    "machine_id",
    "step",
    "failure_mode",
    "degradation_score",
    "is_anomalous",
    "health_state",
    "onset_step",
    "steps_to_failure"
)

ground_truth_pdf = ground_truth_df.toPandas()

print("Ground-truth shape:", ground_truth_pdf.shape)
print("Columns:", list(ground_truth_pdf.columns))

In [0]:
# ============================================================
# CLEAN EVALUATION DATAFRAME
# ============================================================

# Ground truth
eval_spark = spark.table("workspace.default.factorymind_features")

ground_truth_pdf = (
    eval_spark
    .select(
        "machine_id",
        "step",
        "failure_mode",
        "degradation_score",
        "is_anomalous",
        "health_state",
        "onset_step",
        "steps_to_failure"
    )
    .toPandas()
)

# Predictions from the ORIGINAL prediction dataframe.
# Explicitly select only the columns we need.
prediction_pdf = (
    pdf.loc[:, [
        "machine_id",
        "step",
        "anomaly_score",
        "iforest_prediction",
        "iforest_anomaly"
    ]]
    .copy()
)

# Remove duplicate column names if they exist
ground_truth_pdf = ground_truth_pdf.loc[
    :, ~ground_truth_pdf.columns.duplicated()
]

prediction_pdf = prediction_pdf.loc[
    :, ~prediction_pdf.columns.duplicated()
]

print("Ground truth shape:", ground_truth_pdf.shape)
print("Prediction shape:", prediction_pdf.shape)

print("\nGround truth columns:")
print(ground_truth_pdf.columns.tolist())

print("\nPrediction columns:")
print(prediction_pdf.columns.tolist())

In [0]:
# ============================================================
# MERGE GROUND TRUTH + PREDICTIONS
# ============================================================

pdf_eval = pd.merge(
    ground_truth_pdf,
    prediction_pdf,
    on=["machine_id", "step"],
    how="inner",
    validate="one_to_one"
)

print("Evaluation rows:", len(pdf_eval))

print("\nEvaluation columns:")
print(pdf_eval.columns.tolist())

print("\nDuplicate columns:")
print(
    pdf_eval.columns[
        pdf_eval.columns.duplicated()
    ].tolist()
)

In [0]:
# ============================================================
# FAILURE-MODE BENCHMARK
# ============================================================

from sklearn.metrics import precision_score, recall_score

print("=" * 60)
print("ISOLATION FOREST — FAILURE MODE PERFORMANCE")
print("=" * 60)

for mode in sorted(pdf_eval["failure_mode"].dropna().unique()):

    subset = pdf_eval[
        pdf_eval["failure_mode"] == mode
    ]

    y_true = subset["is_anomalous"].astype(int)
    y_pred = subset["iforest_anomaly"].astype(int)

    print(f"\n{mode}")
    print("-" * 40)
    print(f"Rows:       {len(subset):,}")
    print(f"Anomalies:  {y_true.sum():,}")
    print(f"Detected:   {y_pred.sum():,}")

    print(
        f"Recall:     "
        f"{recall_score(y_true, y_pred, zero_division=0):.4f}"
    )

    print(
        f"Precision:  "
        f"{precision_score(y_true, y_pred, zero_division=0):.4f}"
    )

In [0]:
# ============================================================
# FACTORYMIND — ANOMALY DETECTION LEAD TIME
# ============================================================

failed_machines = pdf_eval[
    pdf_eval["failure_mode"] != "none"
]["machine_id"].unique()

lead_times = []

for machine_id in failed_machines:

    machine = pdf_eval[
        pdf_eval["machine_id"] == machine_id
    ].sort_values("step")

    # Ground-truth failure onset
    onset_values = machine["onset_step"].dropna()

    if len(onset_values) == 0:
        continue

    onset = int(onset_values.iloc[0])

    # First anomaly detected BEFORE or AT failure onset
    detections = machine[
        (machine["iforest_anomaly"] == 1) &
        (machine["step"] <= onset)
    ]

    if len(detections) > 0:

        first_detection = int(
            detections["step"].min()
        )

        lead_steps = onset - first_detection

        lead_times.append({
            "machine_id": machine_id,
            "failure_mode": machine["failure_mode"].iloc[0],
            "onset_step": onset,
            "first_detection_step": first_detection,
            "lead_steps": lead_steps,
            "lead_minutes": lead_steps
        })

lead_df = pd.DataFrame(lead_times)

print("=" * 60)
print("FACTORYMIND — DETECTION LEAD TIME")
print("=" * 60)

print(f"Failed machines:       {len(failed_machines)}")
print(f"Detected before onset: {len(lead_df)}")

if len(lead_df) > 0:

    print(
        f"\nMean lead time:   "
        f"{lead_df['lead_minutes'].mean():.2f} minutes"
    )

    print(
        f"Median lead time: "
        f"{lead_df['lead_minutes'].median():.2f} minutes"
    )

    print(
        f"Minimum lead time:"
        f" {lead_df['lead_minutes'].min():.2f} minutes"
    )

    print(
        f"Maximum lead time:"
        f" {lead_df['lead_minutes'].max():.2f} minutes"
    )

    print("\nBy failure mode:")
    print(
        lead_df
        .groupby("failure_mode")["lead_minutes"]
        .agg(["count", "mean", "median", "min", "max"])
        .round(2)
    )

In [0]:
# ============================================================
# COOLING FAILURE — TRAJECTORY ANALYSIS
# ============================================================

cooling = pdf_eval[
    pdf_eval["failure_mode"] == "cooling_failure"
].copy()

for machine_id in cooling["machine_id"].unique():

    m = cooling[
        cooling["machine_id"] == machine_id
    ].sort_values("step")

    onset = int(m["onset_step"].dropna().iloc[0])

    detections_before = m[
        (m["iforest_anomaly"] == 1) &
        (m["step"] <= onset)
    ]

    detections_after = m[
        (m["iforest_anomaly"] == 1) &
        (m["step"] > onset)
    ]

    print("=" * 60)
    print(f"Machine: {machine_id}")
    print(f"Onset:   {onset}")

    if len(detections_before) > 0:
        print(
            "First detection BEFORE onset:",
            detections_before["step"].min()
        )
    else:
        print("No detection before onset")

    if len(detections_after) > 0:
        print(
            "First detection AFTER onset:",
            detections_after["step"].min()
        )

    print("\nLast 10 observations before onset:")
    print(
        m[m["step"] <= onset][
            [
                "step",
                "degradation_score",
                "anomaly_score",
                "iforest_anomaly"
            ]
        ].tail(10).to_string(index=False)
    )

In [0]:
# ============================================================
# FACTORYMIND — RUL DATASET PREPARATION
# ============================================================

import pandas as pd
import numpy as np

# Load feature table
rul_spark = spark.table("workspace.default.factorymind_features")

print("Total rows:", rul_spark.count())

# ------------------------------------------------------------
# Ground-truth target is steps_to_failure.
#
# IMPORTANT:
# steps_to_failure MUST NOT be included in the features.
# ------------------------------------------------------------

GROUND_TRUTH = {
    "degradation_score",
    "health_state",
    "is_anomalous",
    "failure_mode",
    "onset_step",
    "steps_to_failure"
}

IDENTIFIERS = {
    "machine_id",
    "timestamp"
}

feature_columns = [
    c for c in rul_spark.columns
    if c not in GROUND_TRUTH
    and c not in IDENTIFIERS
]

print("Number of input features:", len(feature_columns))

# ------------------------------------------------------------
# Keep only observations where a future failure exists
# ------------------------------------------------------------

rul_df = (
    rul_spark
    .select(
        "machine_id",
        "step",
        "failure_mode",
        "steps_to_failure",
        *feature_columns
    )
    .filter("steps_to_failure IS NOT NULL")
    .dropna(subset=feature_columns)
)

print("RUL rows:", rul_df.count())

# Convert to pandas
rul_pdf = rul_df.toPandas()

print("RUL dataset shape:", rul_pdf.shape)

print("\nFailure-mode distribution:")
print(rul_pdf["failure_mode"].value_counts())

print("\nRUL target statistics:")
print(
    rul_pdf["steps_to_failure"].describe()
)

In [0]:
# ============================================================
# FACTORYMIND — MACHINE-LEVEL RUL TRAIN/TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

# Unique failed machines
machines = rul_pdf["machine_id"].unique()

print("Total failed machines:", len(machines))

# 80/20 split by MACHINE
train_machines, test_machines = train_test_split(
    machines,
    test_size=0.20,
    random_state=42
)

print("Training machines:", len(train_machines))
print("Test machines:", len(test_machines))

# ------------------------------------------------------------
# Create train/test datasets
# ------------------------------------------------------------

train_rul = rul_pdf[
    rul_pdf["machine_id"].isin(train_machines)
].copy()

test_rul = rul_pdf[
    rul_pdf["machine_id"].isin(test_machines)
].copy()

print("\nTraining rows:", len(train_rul))
print("Test rows:", len(test_rul))

print("\nTraining machines:")
print(sorted(train_machines))

print("\nTest machines:")
print(sorted(test_machines))

# ------------------------------------------------------------
# Verify NO machine overlap
# ------------------------------------------------------------

overlap = set(train_machines) & set(test_machines)

print("\nMachine overlap:", overlap)

assert len(overlap) == 0, "DATA LEAKAGE: machine appears in both sets!"

print("SUCCESS: No machine leakage.")

In [0]:
# ============================================================
# RUL MODEL MATRICES
# ============================================================

X_train = train_rul[feature_columns]
y_train = train_rul["steps_to_failure"]

X_test = test_rul[feature_columns]
y_test = test_rul["steps_to_failure"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nTarget range:")
print("Train:", y_train.min(), "→", y_train.max())
print("Test: ", y_test.min(), "→", y_test.max())

In [0]:
# ============================================================
# FACTORYMIND — RUL BASELINE MODEL
# ============================================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

# ------------------------------------------------------------
# Remove temporal identifier
# ------------------------------------------------------------

rul_feature_columns = [
    c for c in feature_columns
    if c != "step"
]

X_train = train_rul[rul_feature_columns]
y_train = train_rul["steps_to_failure"]

X_test = test_rul[rul_feature_columns]
y_test = test_rul["steps_to_failure"]

print("RUL features:", len(rul_feature_columns))
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

# ------------------------------------------------------------
# Train Random Forest baseline
# ------------------------------------------------------------

rul_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=3,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

print("\nTraining RUL model...")

rul_model.fit(
    X_train,
    y_train
)

print("SUCCESS: RUL model trained")

# ------------------------------------------------------------
# Predict
# ------------------------------------------------------------

y_pred = rul_model.predict(X_test)

# RUL cannot be negative
y_pred = np.maximum(y_pred, 0)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n" + "=" * 60)
print("FACTORYMIND — RUL BASELINE RESULTS")
print("=" * 60)

print(f"MAE:  {mae:.2f} steps")
print(f"RMSE: {rmse:.2f} steps")
print(f"R²:   {r2:.4f}")

print("\nApproximate time interpretation:")
print(f"MAE:  {mae:.2f} minutes")
print(f"RMSE: {rmse:.2f} minutes")

In [0]:
# ============================================================
# FACTORYMIND — RUL ERROR BY REMAINING-LIFE WINDOW
# ============================================================

eval_rul = test_rul[
    ["machine_id", "failure_mode", "steps_to_failure"]
].copy()

eval_rul["actual_rul"] = y_test.values
eval_rul["predicted_rul"] = y_pred
eval_rul["absolute_error"] = (
    np.abs(
        eval_rul["actual_rul"]
        - eval_rul["predicted_rul"]
    )
)

# ------------------------------------------------------------
# Define operational RUL windows
# ------------------------------------------------------------

def rul_window(x):
    if x <= 30:
        return "0-30 min"
    elif x <= 60:
        return "31-60 min"
    elif x <= 120:
        return "61-120 min"
    else:
        return ">120 min"

eval_rul["rul_window"] = (
    eval_rul["actual_rul"]
    .apply(rul_window)
)

# ------------------------------------------------------------
# Error by RUL window
# ------------------------------------------------------------

window_results = (
    eval_rul
    .groupby("rul_window", observed=True)
    .agg(
        samples=("actual_rul", "size"),
        mae=("absolute_error", "mean"),
        actual_mean=("actual_rul", "mean"),
        predicted_mean=("predicted_rul", "mean")
    )
    .round(2)
)

print("=" * 60)
print("RUL ERROR BY REMAINING-LIFE WINDOW")
print("=" * 60)

print(window_results)

# ------------------------------------------------------------
# Error by failure mode
# ------------------------------------------------------------

mode_results = (
    eval_rul
    .groupby("failure_mode")
    .agg(
        samples=("actual_rul", "size"),
        mae=("absolute_error", "mean"),
        actual_mean=("actual_rul", "mean"),
        predicted_mean=("predicted_rul", "mean")
    )
    .round(2)
)

print("\n" + "=" * 60)
print("RUL ERROR BY FAILURE MODE")
print("=" * 60)

print(mode_results)

In [0]:
# ============================================================
# SAMPLE RUL PREDICTIONS
# ============================================================

display(
    eval_rul[
        [
            "machine_id",
            "failure_mode",
            "actual_rul",
            "predicted_rul",
            "absolute_error"
        ]
    ]
    .sort_values("actual_rul")
    .head(30)
)

In [0]:
# ============================================================
# RUL PERFORMANCE — CRITICAL WINDOWS
# ============================================================

# Make sure actual/predicted values are numeric
eval_rul["actual_rul"] = pd.to_numeric(
    eval_rul["actual_rul"]
)

eval_rul["predicted_rul"] = pd.to_numeric(
    eval_rul["predicted_rul"]
)

eval_rul["absolute_error"] = (
    eval_rul["actual_rul"]
    - eval_rul["predicted_rul"]
).abs()

# ------------------------------------------------------------
# Critical RUL windows
# ------------------------------------------------------------

windows = [
    ("0-30 min", 0, 30),
    ("31-60 min", 31, 60),
    ("61-120 min", 61, 120),
    (">120 min", 121, np.inf)
]

print("=" * 70)
print("FACTORYMIND — RUL PERFORMANCE BY REMAINING LIFE")
print("=" * 70)

for name, lower, upper in windows:

    subset = eval_rul[
        (eval_rul["actual_rul"] >= lower) &
        (eval_rul["actual_rul"] <= upper)
    ]

    if len(subset) == 0:
        continue

    mae = subset["absolute_error"].mean()

    rmse = np.sqrt(
        np.mean(
            subset["absolute_error"] ** 2
        )
    )

    print(f"\n{name}")
    print("-" * 40)
    print(f"Samples:          {len(subset):,}")
    print(f"Mean actual RUL:  {subset['actual_rul'].mean():.2f}")
    print(f"Mean predicted:   {subset['predicted_rul'].mean():.2f}")
    print(f"MAE:              {mae:.2f} min")
    print(f"RMSE:             {rmse:.2f} min")

In [0]:
# ============================================================
# RUL PERFORMANCE BY FAILURE MODE
# ============================================================

print("=" * 70)
print("FACTORYMIND — RUL PERFORMANCE BY FAILURE MODE")
print("=" * 70)

for mode in sorted(eval_rul["failure_mode"].unique()):

    subset = eval_rul[
        eval_rul["failure_mode"] == mode
    ]

    mae = subset["absolute_error"].mean()

    rmse = np.sqrt(
        np.mean(
            subset["absolute_error"] ** 2
        )
    )

    print(f"\n{mode}")
    print("-" * 40)
    print(f"Samples:          {len(subset):,}")
    print(f"Mean actual RUL:  {subset['actual_rul'].mean():.2f}")
    print(f"Mean predicted:   {subset['predicted_rul'].mean():.2f}")
    print(f"MAE:              {mae:.2f} min")
    print(f"RMSE:             {rmse:.2f} min")

In [0]:
# ============================================================
# FACTORYMIND — WORST RUL PREDICTIONS
# ============================================================

worst = (
    eval_rul
    .sort_values("absolute_error", ascending=False)
    .head(30)
)

print("=" * 70)
print("30 WORST RUL PREDICTIONS")
print("=" * 70)

print(
    worst[
        [
            "machine_id",
            "failure_mode",
            "actual_rul",
            "predicted_rul",
            "absolute_error"
        ]
    ].to_string(index=False)
)

In [0]:
# ============================================================
# RUL BIAS ANALYSIS
# ============================================================

eval_rul["error"] = (
    eval_rul["predicted_rul"]
    - eval_rul["actual_rul"]
)

print("=" * 70)
print("RUL BIAS BY FAILURE MODE")
print("=" * 70)

bias = (
    eval_rul
    .groupby("failure_mode")
    .agg(
        samples=("error", "size"),
        mean_error=("error", "mean"),
        median_error=("error", "median"),
        mae=("absolute_error", "mean")
    )
    .round(2)
)

print(bias)

In [0]:
# ============================================================
# RUL ERROR BY REMAINING-LIFE RANGE
# ============================================================

eval_rul["rul_bucket"] = pd.cut(
    eval_rul["actual_rul"],
    bins=[-1, 30, 60, 120, 240, 360, 500],
    labels=[
        "0-30",
        "31-60",
        "61-120",
        "121-240",
        "241-360",
        "361-500"
    ]
)

range_results = (
    eval_rul
    .groupby("rul_bucket", observed=False)
    .agg(
        samples=("actual_rul", "size"),
        actual_mean=("actual_rul", "mean"),
        predicted_mean=("predicted_rul", "mean"),
        mae=("absolute_error", "mean"),
        bias=("error", "mean")
    )
    .round(2)
)

print("=" * 70)
print("FACTORYMIND — RUL PERFORMANCE BY REMAINING LIFE")
print("=" * 70)

print(range_results)

In [0]:
# ============================================================
# RUL V2 — TRAINING TARGET DISTRIBUTION
# ============================================================

print("=" * 70)
print("RUL TRAINING TARGET DISTRIBUTION")
print("=" * 70)

train_rul["rul_bucket"] = pd.cut(
    train_rul["steps_to_failure"],
    bins=[-1, 30, 60, 120, 240, 360, 500],
    labels=[
        "0-30",
        "31-60",
        "61-120",
        "121-240",
        "241-360",
        "361-500"
    ]
)

print(
    train_rul["rul_bucket"]
    .value_counts()
    .sort_index()
)

print("\nPercentages:")
print(
    (
        train_rul["rul_bucket"]
        .value_counts(normalize=True)
        .sort_index()
        * 100
    ).round(2)
)

In [0]:
# ============================================================
# FACTORYMIND — RUL V2: WEIGHTED RANDOM FOREST
# ============================================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

# ------------------------------------------------------------
# Features
# ------------------------------------------------------------

rul_feature_columns = [
    c for c in feature_columns
    if c != "step"
]

X_train_v2 = train_rul[rul_feature_columns]
y_train_v2 = train_rul["steps_to_failure"]

X_test_v2 = test_rul[rul_feature_columns]
y_test_v2 = test_rul["steps_to_failure"]

# ------------------------------------------------------------
# Create balanced sample weights
#
# Rare RUL ranges receive higher importance.
# ------------------------------------------------------------

def calculate_rul_weight(rul):

    if rul <= 30:
        return 1.0

    elif rul <= 60:
        return 3.0

    elif rul <= 120:
        return 2.5

    elif rul <= 240:
        return 2.0

    elif rul <= 360:
        return 3.0

    else:
        return 5.0


sample_weights = y_train_v2.apply(
    calculate_rul_weight
).values

print("=" * 60)
print("RUL V2 SAMPLE WEIGHTS")
print("=" * 60)

weight_df = pd.DataFrame({
    "rul": y_train_v2,
    "weight": sample_weights
})

print(
    weight_df
    .groupby(
        pd.cut(
            weight_df["rul"],
            bins=[-1, 30, 60, 120, 240, 360, 500],
            labels=[
                "0-30",
                "31-60",
                "61-120",
                "121-240",
                "241-360",
                "361-500"
            ]
        ),
        observed=False
    )["weight"]
    .agg(["count", "mean"])
)

# ------------------------------------------------------------
# Train weighted Random Forest
# ------------------------------------------------------------

rul_model_v2 = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    min_samples_leaf=3,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

print("\nTraining weighted RUL model...")

rul_model_v2.fit(
    X_train_v2,
    y_train_v2,
    sample_weight=sample_weights
)

print("SUCCESS: RUL V2 trained")

# ------------------------------------------------------------
# Predict
# ------------------------------------------------------------

y_pred_v2 = rul_model_v2.predict(X_test_v2)

# RUL cannot be negative
y_pred_v2 = np.maximum(y_pred_v2, 0)

# ------------------------------------------------------------
# Overall metrics
# ------------------------------------------------------------

mae_v2 = mean_absolute_error(
    y_test_v2,
    y_pred_v2
)

rmse_v2 = np.sqrt(
    mean_squared_error(
        y_test_v2,
        y_pred_v2
    )
)

r2_v2 = r2_score(
    y_test_v2,
    y_pred_v2
)

print("\n" + "=" * 60)
print("FACTORYMIND — RUL V2 RESULTS")
print("=" * 60)

print(f"MAE:  {mae_v2:.2f} minutes")
print(f"RMSE: {rmse_v2:.2f} minutes")
print(f"R²:   {r2_v2:.4f}")

In [0]:
# ============================================================
# RUL V2 — ERROR BY REMAINING-LIFE RANGE
# ============================================================

eval_v2 = test_rul[
    ["machine_id", "failure_mode", "steps_to_failure"]
].copy()

eval_v2["actual_rul"] = y_test_v2.values
eval_v2["predicted_rul"] = y_pred_v2

eval_v2["absolute_error"] = (
    eval_v2["actual_rul"]
    - eval_v2["predicted_rul"]
).abs()

eval_v2["error"] = (
    eval_v2["predicted_rul"]
    - eval_v2["actual_rul"]
)

eval_v2["rul_bucket"] = pd.cut(
    eval_v2["actual_rul"],
    bins=[-1, 30, 60, 120, 240, 360, 500],
    labels=[
        "0-30",
        "31-60",
        "61-120",
        "121-240",
        "241-360",
        "361-500"
    ]
)

v2_range = (
    eval_v2
    .groupby("rul_bucket", observed=False)
    .agg(
        samples=("actual_rul", "size"),
        actual_mean=("actual_rul", "mean"),
        predicted_mean=("predicted_rul", "mean"),
        mae=("absolute_error", "mean"),
        bias=("error", "mean")
    )
    .round(2)
)

print("=" * 70)
print("RUL V2 — PERFORMANCE BY REMAINING LIFE")
print("=" * 70)

print(v2_range)

In [0]:
# ============================================================
# V1 vs V2 — RANGE COMPARISON
# ============================================================

v1_range = (
    eval_rul
    .groupby("rul_bucket", observed=False)
    .agg(
        v1_mae=("absolute_error", "mean"),
        v1_bias=("error", "mean")
    )
)

v2_range_compare = (
    eval_v2
    .groupby("rul_bucket", observed=False)
    .agg(
        v2_mae=("absolute_error", "mean"),
        v2_bias=("error", "mean")
    )
)

comparison_range = pd.concat(
    [v1_range, v2_range_compare],
    axis=1
).round(2)

print(comparison_range)

In [0]:
# ============================================================
# FACTORYMIND — RUL V3 TRAJECTORY FEATURES
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load Silver features
rul_source = spark.table(
    "workspace.default.factorymind_features"
)

# ------------------------------------------------------------
# Sensor columns
# ------------------------------------------------------------

sensors = [
    "temperature",
    "vibration",
    "pressure",
    "motor_current",
    "rpm",
    "load",
    "coolant_temperature",
    "coolant_flow",
    "acoustic_level"
]

# ------------------------------------------------------------
# Machine/time window
# ------------------------------------------------------------

w = (
    Window
    .partitionBy("machine_id")
    .orderBy("step")
)

w10 = w.rowsBetween(-10, 0)
w30 = w.rowsBetween(-30, 0)

trajectory_df = rul_source

# ------------------------------------------------------------
# Add trajectory features
# ------------------------------------------------------------

for sensor in sensors:

    # Recent means
    trajectory_df = trajectory_df.withColumn(
        f"{sensor}_mean_10",
        F.avg(F.col(sensor)).over(w10)
    )

    trajectory_df = trajectory_df.withColumn(
        f"{sensor}_mean_30",
        F.avg(F.col(sensor)).over(w30)
    )

    # Short-term change
    trajectory_df = trajectory_df.withColumn(
        f"{sensor}_delta_10",
        F.col(sensor) - F.lag(sensor, 10).over(w)
    )

    trajectory_df = trajectory_df.withColumn(
        f"{sensor}_delta_30",
        F.col(sensor) - F.lag(sensor, 30).over(w)
    )

    # Volatility
    trajectory_df = trajectory_df.withColumn(
        f"{sensor}_std_10",
        F.stddev(F.col(sensor)).over(w10)
    )

    # Approximate slope
    trajectory_df = trajectory_df.withColumn(
        f"{sensor}_slope_10",
        (
            F.col(sensor)
            - F.lag(sensor, 10).over(w)
        ) / 10.0
    )

    trajectory_df = trajectory_df.withColumn(
        f"{sensor}_slope_30",
        (
            F.col(sensor)
            - F.lag(sensor, 30).over(w)
        ) / 30.0
    )

# ------------------------------------------------------------
# Keep evaluation labels, but never use them as features
# ------------------------------------------------------------

trajectory_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "workspace.default.factorymind_rul_features"
    )

print("SUCCESS")
print(
    "Rows:",
    trajectory_df.count()
)

print(
    "Columns:",
    len(trajectory_df.columns)
)

In [0]:
# ============================================================
# FACTORYMIND — RUL V3 DATASET PREPARATION
# ============================================================

rul_v3_spark = spark.table(
    "workspace.default.factorymind_rul_features"
)

GROUND_TRUTH = {
    "degradation_score",
    "health_state",
    "is_anomalous",
    "failure_mode",
    "onset_step",
    "steps_to_failure"
}

IDENTIFIERS = {
    "machine_id",
    "timestamp"
}

# Never use ground-truth or identifiers as model inputs
rul_v3_features = [
    c for c in rul_v3_spark.columns
    if c not in GROUND_TRUTH
    and c not in IDENTIFIERS
    and c != "step"
]

print("Total V3 features:", len(rul_v3_features))

# Keep only rows with an RUL target
rul_v3_df = (
    rul_v3_spark
    .select(
        "machine_id",
        "step",
        "failure_mode",
        "steps_to_failure",
        *rul_v3_features
    )
    .filter("steps_to_failure IS NOT NULL")
    .dropna(subset=rul_v3_features)
)

print("V3 rows:", rul_v3_df.count())

rul_v3_pdf = rul_v3_df.toPandas()

print("V3 shape:", rul_v3_pdf.shape)

In [0]:
# ============================================================
# SAME MACHINE SPLIT AS RUL V1/V2
# ============================================================

train_v3 = rul_v3_pdf[
    rul_v3_pdf["machine_id"].isin(train_machines)
].copy()

test_v3 = rul_v3_pdf[
    rul_v3_pdf["machine_id"].isin(test_machines)
].copy()

print("Training rows:", len(train_v3))
print("Test rows:", len(test_v3))

print("Training machines:", train_v3["machine_id"].nunique())
print("Test machines:", test_v3["machine_id"].nunique())

# Verify leakage
overlap_v3 = (
    set(train_v3["machine_id"])
    &
    set(test_v3["machine_id"])
)

print("Machine overlap:", overlap_v3)

assert len(overlap_v3) == 0

print("SUCCESS: V3 has no machine leakage.")

In [0]:
# ============================================================
# FACTORYMIND — RUL V3 GRADIENT BOOSTING MODEL
# ============================================================

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

X_train_v3 = train_v3[rul_v3_features]
y_train_v3 = train_v3["steps_to_failure"]

X_test_v3 = test_v3[rul_v3_features]
y_test_v3 = test_v3["steps_to_failure"]

print("X_train:", X_train_v3.shape)
print("X_test:", X_test_v3.shape)

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

rul_model_v3 = HistGradientBoostingRegressor(
    max_iter=400,
    learning_rate=0.05,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=1.0,
    random_state=42
)

print("\nTraining RUL V3...")

rul_model_v3.fit(
    X_train_v3,
    y_train_v3
)

print("SUCCESS: RUL V3 trained")

# ------------------------------------------------------------
# Predict
# ------------------------------------------------------------

y_pred_v3 = rul_model_v3.predict(X_test_v3)

y_pred_v3 = np.maximum(
    y_pred_v3,
    0
)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

mae_v3 = mean_absolute_error(
    y_test_v3,
    y_pred_v3
)

rmse_v3 = np.sqrt(
    mean_squared_error(
        y_test_v3,
        y_pred_v3
    )
)

r2_v3 = r2_score(
    y_test_v3,
    y_pred_v3
)

print("\n" + "=" * 60)
print("FACTORYMIND — RUL V3 RESULTS")
print("=" * 60)

print(f"MAE:  {mae_v3:.2f} minutes")
print(f"RMSE: {rmse_v3:.2f} minutes")
print(f"R²:   {r2_v3:.4f}")

In [0]:
# ============================================================
# FACTORYMIND — RUL V1 vs V3 BY REMAINING LIFE
# ============================================================

eval_v3 = test_v3[
    ["machine_id", "failure_mode", "steps_to_failure"]
].copy()

eval_v3["actual_rul"] = y_test_v3.values
eval_v3["predicted_rul"] = y_pred_v3

eval_v3["absolute_error"] = (
    eval_v3["actual_rul"]
    - eval_v3["predicted_rul"]
).abs()

eval_v3["error"] = (
    eval_v3["predicted_rul"]
    - eval_v3["actual_rul"]
)

eval_v3["rul_bucket"] = pd.cut(
    eval_v3["actual_rul"],
    bins=[-1, 30, 60, 120, 240, 360, 500],
    labels=[
        "0-30",
        "31-60",
        "61-120",
        "121-240",
        "241-360",
        "361-500"
    ]
)

v3_range = (
    eval_v3
    .groupby("rul_bucket", observed=False)
    .agg(
        samples=("actual_rul", "size"),
        actual_mean=("actual_rul", "mean"),
        predicted_mean=("predicted_rul", "mean"),
        mae=("absolute_error", "mean"),
        bias=("error", "mean")
    )
    .round(2)
)

print("=" * 70)
print("RUL V3 — PERFORMANCE BY REMAINING LIFE")
print("=" * 70)

print(v3_range)

In [0]:
# ============================================================
# V1 vs V3 RANGE COMPARISON
# ============================================================

v1_range = (
    eval_rul
    .groupby("rul_bucket", observed=False)
    .agg(
        v1_mae=("absolute_error", "mean"),
        v1_bias=("error", "mean")
    )
)

v3_range_compare = (
    eval_v3
    .groupby("rul_bucket", observed=False)
    .agg(
        v3_mae=("absolute_error", "mean"),
        v3_bias=("error", "mean")
    )
)

comparison_v1_v3 = pd.concat(
    [v1_range, v3_range_compare],
    axis=1
).round(2)

print("=" * 70)
print("RUL V1 vs V3")
print("=" * 70)

print(comparison_v1_v3)

In [0]:
# ============================================================
# DIAGNOSTIC — DEGRADATION SCORE vs RUL
# ============================================================

diagnostic = rul_v3_spark.select(
    "machine_id",
    "step",
    "failure_mode",
    "degradation_score",
    "steps_to_failure"
).filter(
    "steps_to_failure IS NOT NULL"
).toPandas()

print("=" * 70)
print("DEGRADATION SCORE → RUL DIAGNOSTIC")
print("=" * 70)

print(
    diagnostic[
        [
            "degradation_score",
            "steps_to_failure"
        ]
    ].corr()
)

print("\nAverage RUL by degradation stage:")

diagnostic["degradation_bucket"] = pd.cut(
    diagnostic["degradation_score"],
    bins=[
        -0.001,
        0.05,
        0.20,
        0.40,
        0.60,
        0.80,
        1.0
    ],
    labels=[
        "0-0.05",
        "0.05-0.20",
        "0.20-0.40",
        "0.40-0.60",
        "0.60-0.80",
        "0.80-1.00"
    ]
)

print(
    diagnostic
    .groupby(
        "degradation_bucket",
        observed=False
    )["steps_to_failure"]
    .agg(
        ["count", "mean", "median", "min", "max"]
    )
    .round(2)
)

In [0]:
# ============================================================
# FACTORYMIND — INSPECT RUL GROUND TRUTH
# ============================================================

inspect_cols = [
    "machine_id",
    "step",
    "failure_mode",
    "degradation_score",
    "health_state",
    "onset_step",
    "steps_to_failure"
]

inspect_df = (
    spark.table("workspace.default.factorymind_features")
    .select(*inspect_cols)
    .filter("failure_mode != 'none'")
    .orderBy("machine_id", "step")
    .toPandas()
)

# ------------------------------------------------------------
# Inspect one complete failed machine
# ------------------------------------------------------------

machine = (
    inspect_df[
        inspect_df["machine_id"] == "M-0006"
    ]
    .copy()
)

print("=" * 80)
print("M-0006 — COMPLETE RUL TARGET TRAJECTORY")
print("=" * 80)

print(
    machine[
        [
            "step",
            "degradation_score",
            "health_state",
            "onset_step",
            "steps_to_failure"
        ]
    ].to_string(index=False)
)

In [0]:
# ============================================================
# RUL TARGET TRANSITIONS
# ============================================================

print("=" * 80)
print("RUL TARGET TRANSITIONS")
print("=" * 80)

summary = (
    inspect_df
    .groupby("failure_mode")
    .agg(
        rows=("steps_to_failure", "size"),
        rul_min=("steps_to_failure", "min"),
        rul_max=("steps_to_failure", "max"),
        rul_mean=("steps_to_failure", "mean"),
        degradation_min=("degradation_score", "min"),
        degradation_max=("degradation_score", "max")
    )
    .round(2)
)

print(summary)

In [0]:
# ============================================================
# RUL TARGET TRANSITIONS
# ============================================================

print("=" * 80)
print("RUL TARGET TRANSITIONS")
print("=" * 80)

summary = (
    inspect_df
    .groupby("failure_mode")
    .agg(
        rows=("steps_to_failure", "size"),
        rul_min=("steps_to_failure", "min"),
        rul_max=("steps_to_failure", "max"),
        rul_mean=("steps_to_failure", "mean"),
        degradation_min=("degradation_score", "min"),
        degradation_max=("degradation_score", "max")
    )
    .round(2)
)

print(summary)

In [0]:
# ============================================================
# HOW RUL VALUES RELATE TO DEGRADATION
# ============================================================

print("=" * 80)
print("NON-ZERO RUL BY DEGRADATION")
print("=" * 80)

nonzero = inspect_df[
    inspect_df["steps_to_failure"] > 0
]

print(
    nonzero[
        [
            "degradation_score",
            "steps_to_failure"
        ]
    ].describe()
)

print(
    "\nMaximum degradation where RUL > 0:",
    nonzero["degradation_score"].max()
)

In [0]:
from pyspark.sql import functions as F

df = (
    spark.table("workspace.default.factorymind_telemetry")
    .join(
        spark.table("workspace.default.factorymind_machine_labels"),
        on="machine_id",
        how="left"
    )
)

print("Columns:")
print(df.columns)

print("\nHealth states:")
(
    df.groupBy("health_state")
      .count()
      .orderBy("health_state")
      .show(truncate=False)
)

In [0]:
from pyspark.sql import functions as F

telemetry = spark.table("workspace.default.factorymind_telemetry").alias("t")
labels = spark.table("workspace.default.factorymind_machine_labels").alias("m")

df = (
    telemetry
    .join(
        labels,
        F.col("t.machine_id") == F.col("m.machine_id"),
        "left"
    )
    .select(
        F.col("t.machine_id").alias("machine_id"),
        F.col("t.step").alias("step"),
        F.col("t.failure_mode").alias("failure_mode"),
        F.col("t.degradation_score").alias("degradation_score"),
        F.col("t.health_state").alias("health_state")
    )
)

print("Columns:")
print(df.columns)

print("\nHealth states:")
(
    df.groupBy("health_state")
      .count()
      .orderBy("health_state")
      .show(truncate=False)
)

In [0]:
print("=" * 80)
print("FAILED MACHINE TRAJECTORY — M-0006")
print("=" * 80)

machine = (
    df
    .filter(F.col("machine_id") == "M-0006")
    .filter(
        (F.col("step") % 25 == 0) |
        (F.col("degradation_score") > 0)
    )
    .orderBy("step")
)

machine.show(100, truncate=False)

In [0]:
print("=" * 80)
print("FAILURE ENDPOINTS")
print("=" * 80)

failure_endpoints = (
    df
    .filter(F.col("health_state") == "critical_failure")
    .groupBy("machine_id")
    .agg(
        F.min("step").alias("failure_endpoint_step"),
        F.first("failure_mode").alias("failure_mode")
    )
    .orderBy("machine_id")
)

failure_endpoints.show(50, truncate=False)

In [0]:
from pyspark.sql import functions as F

# ============================================================
# CORRECT RUL TARGET
# ============================================================

features = spark.table("workspace.default.factorymind_features").alias("f")

# First critical-failure step for every failed machine
endpoints = (
    spark.table("workspace.default.factorymind_telemetry")
    .filter(F.col("health_state") == "critical_failure")
    .groupBy("machine_id")
    .agg(
        F.min("step").alias("failure_endpoint_step")
    )
    .alias("e")
)

# Join endpoint to every feature row
rul_df = (
    features
    .join(
        endpoints,
        on="machine_id",
        how="inner"
    )
    # Only use observations up to the actual failure
    .filter(
        F.col("step") <= F.col("failure_endpoint_step")
    )
    # TRUE remaining useful life
    .withColumn(
        "rul",
        F.col("failure_endpoint_step") - F.col("step")
    )
)

print("=" * 80)
print("CORRECTED RUL DATASET")
print("=" * 80)

print("Rows:", rul_df.count())

rul_df.select(
    "machine_id",
    "step",
    "failure_endpoint_step",
    "rul",
    "failure_mode",
    "degradation_score",
    "health_state"
).orderBy("machine_id", "step").show(30, truncate=False)

In [0]:
print("=" * 80)
print("RUL TARGET VALIDATION")
print("=" * 80)

rul_df.select("rul").describe().show()

print("\nRUL by degradation:")
(
    rul_df
    .groupBy(
        F.when(F.col("degradation_score") == 0, "healthy")
         .when(F.col("degradation_score") < 0.2, "0.0-0.2")
         .when(F.col("degradation_score") < 0.4, "0.2-0.4")
         .when(F.col("degradation_score") < 0.6, "0.4-0.6")
         .when(F.col("degradation_score") < 0.8, "0.6-0.8")
         .otherwise("0.8-1.0")
         .alias("degradation_stage")
    )
    .agg(
        F.count("*").alias("rows"),
        F.round(F.mean("rul"), 2).alias("mean_rul"),
        F.round(F.min("rul"), 2).alias("min_rul"),
        F.round(F.max("rul"), 2).alias("max_rul")
    )
    .orderBy("degradation_stage")
    .show(truncate=False)
)

In [0]:
(
    rul_df
    .filter(F.col("machine_id") == "M-0006")
    .select(
        "machine_id",
        "step",
        "degradation_score",
        "health_state",
        "failure_endpoint_step",
        "rul"
    )
    .filter(
        (F.col("step") % 50 == 0) |
        (F.col("step") >= 700)
    )
    .orderBy("step")
    .show(100, truncate=False)
)

In [0]:
# ============================================================
# SAVE CORRECTED RUL DATASET
# ============================================================

RUL_TABLE = "workspace.default.factorymind_rul"

(
    rul_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(RUL_TABLE)
)

print(f"Saved: {RUL_TABLE}")

print("Rows:", spark.table(RUL_TABLE).count())
print("Columns:", len(spark.table(RUL_TABLE).columns))

In [0]:
import pandas as pd
import numpy as np

rul_spark = spark.table("workspace.default.factorymind_rul")

failed_machines = [
    r["machine_id"]
    for r in (
        rul_spark
        .select("machine_id")
        .distinct()
        .orderBy("machine_id")
        .collect()
    )
]

print("Failed machines:", len(failed_machines))
print(failed_machines)

rng = np.random.default_rng(42)

shuffled = failed_machines.copy()
rng.shuffle(shuffled)

n_test = 6

test_machines = sorted(shuffled[:n_test])
train_machines = sorted(shuffled[n_test:])

print("\nTRAIN MACHINES:", len(train_machines))
print(train_machines)

print("\nTEST MACHINES:", len(test_machines))
print(test_machines)

print(
    "\nOverlap:",
    set(train_machines) & set(test_machines)
)

In [0]:
import pandas as pd
import numpy as np

rul_pd = (
    spark.table("workspace.default.factorymind_rul")
    .toPandas()
)

print("Raw shape:", rul_pd.shape)

# Columns that must NOT be model inputs
exclude_cols = {
    "machine_id",
    "step",
    "rul",
    "failure_endpoint_step",
    "steps_to_failure",
    "onset_step",
    "failure_mode",
    "health_state",
    "degradation_score",
}

feature_cols = [
    c for c in rul_pd.columns
    if c not in exclude_cols
]

print("Number of candidate features:", len(feature_cols))
print("\nFeatures:")
print(feature_cols)

In [0]:
X = rul_pd[feature_cols].copy()
y = rul_pd["rul"].astype(float)

# Keep only numeric features
numeric_features = X.select_dtypes(
    include=[np.number]
).columns.tolist()

X = X[numeric_features]

print("Numeric features:", len(numeric_features))
print("X shape:", X.shape)
print("y shape:", y.shape)

# Replace infinities
X = X.replace([np.inf, -np.inf], np.nan)

# Median imputation
X = X.fillna(X.median())

print("Remaining NaNs:", X.isna().sum().sum())

In [0]:
train_mask = rul_pd["machine_id"].isin(train_machines)
test_mask = rul_pd["machine_id"].isin(test_machines)

X_train = X.loc[train_mask].copy()
X_test  = X.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_test  = y.loc[test_mask].copy()

print("=" * 80)
print("RUL TRAIN / TEST SPLIT")
print("=" * 80)

print("Train rows:", len(X_train))
print("Test rows :", len(X_test))

print("Train machines:", rul_pd.loc[train_mask, "machine_id"].nunique())
print("Test machines :", rul_pd.loc[test_mask, "machine_id"].nunique())

print(
    "Machine overlap:",
    set(rul_pd.loc[train_mask, "machine_id"])
    &
    set(rul_pd.loc[test_mask, "machine_id"])
)

In [0]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

rul_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=3,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rul_model.fit(X_train, y_train)

pred = rul_model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print("=" * 80)
print("FACTORYMIND RUL V4")
print("=" * 80)

print(f"MAE  : {mae:.2f} steps")
print(f"RMSE : {rmse:.2f} steps")
print(f"R²   : {r2:.4f}")

In [0]:
eval_df = pd.DataFrame({
    "machine_id": rul_pd.loc[test_mask, "machine_id"].values,
    "failure_mode": rul_pd.loc[test_mask, "failure_mode"].values,
    "actual_rul": y_test.values,
    "predicted_rul": pred
})

eval_df["error"] = (
    eval_df["predicted_rul"] -
    eval_df["actual_rul"]
)

eval_df["abs_error"] = eval_df["error"].abs()

eval_df["rul_range"] = pd.cut(
    eval_df["actual_rul"],
    bins=[-1, 30, 60, 120, 240, 360, 500, 1000],
    labels=[
        "0-30",
        "31-60",
        "61-120",
        "121-240",
        "241-360",
        "361-500",
        "501+"
    ]
)

range_results = (
    eval_df
    .groupby("rul_range", observed=False)
    .agg(
        samples=("actual_rul", "size"),
        actual_mean=("actual_rul", "mean"),
        predicted_mean=("predicted_rul", "mean"),
        MAE=("abs_error", "mean"),
        bias=("error", "mean")
    )
    .round(2)
)

print(range_results)

In [0]:
mode_results = (
    eval_df
    .groupby("failure_mode")
    .agg(
        samples=("actual_rul", "size"),
        actual_mean=("actual_rul", "mean"),
        predicted_mean=("predicted_rul", "mean"),
        MAE=("abs_error", "mean"),
        RMSE=(
            "error",
            lambda x: np.sqrt(np.mean(x ** 2))
        ),
        bias=("error", "mean")
    )
    .round(2)
)

print("=" * 80)
print("RUL PERFORMANCE BY FAILURE MODE")
print("=" * 80)

print(mode_results)

In [0]:
print("=" * 80)
print("RUL PREDICTION DIAGNOSTICS")
print("=" * 80)

print("Correlation:", 
      eval_df["actual_rul"].corr(eval_df["predicted_rul"]))

print("\nPrediction statistics:")
print(
    eval_df[
        ["actual_rul", "predicted_rul", "error", "abs_error"]
    ].describe().round(2)
)

In [0]:
machine_results = (
    eval_df
    .groupby("machine_id")
    .agg(
        samples=("actual_rul", "size"),
        actual_mean=("actual_rul", "mean"),
        predicted_mean=("predicted_rul", "mean"),
        MAE=("abs_error", "mean"),
        RMSE=("error", lambda x: np.sqrt(np.mean(x ** 2))),
        bias=("error", "mean")
    )
    .round(2)
)

print("=" * 80)
print("RUL PERFORMANCE BY TEST MACHINE")
print("=" * 80)

print(machine_results)

In [0]:
for machine_id in sorted(eval_df["machine_id"].unique()):
    
    m = eval_df[
        eval_df["machine_id"] == machine_id
    ].copy()
    
    # Recover step
    m["step"] = rul_pd.loc[
        test_mask & 
        (rul_pd["machine_id"] == machine_id),
        "step"
    ].values
    
    m = m.sort_values("step")
    
    print("\n" + "=" * 60)
    print(machine_id)
    print("=" * 60)
    
    print(
        m[
            ["step", "actual_rul", "predicted_rul"]
        ].iloc[::max(1, len(m)//10)].round(1).to_string(index=False)
    )

In [0]:
from collections import defaultdict
import numpy as np

machine_modes = (
    rul_spark
    .select("machine_id", "failure_mode")
    .distinct()
    .orderBy("machine_id")
    .toPandas()
)

print(machine_modes)

rng = np.random.default_rng(42)

test_machines = []

for mode, group in machine_modes.groupby("failure_mode"):
    machines = sorted(group["machine_id"].tolist())

    # At least one test machine from every failure mode
    selected = rng.choice(machines, size=1, replace=False)[0]
    test_machines.append(selected)

test_machines = sorted(test_machines)

# We now have 6 test machines, exactly one per failure mode.
train_machines = sorted(
    set(failed_machines) - set(test_machines)
)

print("=" * 80)
print("STRATIFIED MACHINE SPLIT")
print("=" * 80)

print("TEST:", test_machines)
print("TRAIN:", train_machines)

print("\nTest failure modes:")
print(
    machine_modes[
        machine_modes["machine_id"].isin(test_machines)
    ].sort_values("failure_mode").to_string(index=False)
)

print("\nTrain failure modes:")
print(
    machine_modes[
        machine_modes["machine_id"].isin(train_machines)
    ]
    .groupby("failure_mode")
    .size()
)

In [0]:
# ============================================================
# STRATIFIED 24/6 MACHINE SPLIT
# Exactly 1 test machine per failure mode
# ============================================================

import numpy as np
import pandas as pd

machine_modes = (
    rul_spark
    .select("machine_id", "failure_mode")
    .distinct()
    .orderBy("machine_id")
    .toPandas()
)

rng = np.random.default_rng(42)

test_machines = []

for mode in sorted(machine_modes["failure_mode"].unique()):
    machines = sorted(
        machine_modes.loc[
            machine_modes["failure_mode"] == mode,
            "machine_id"
        ].tolist()
    )

    selected = rng.choice(machines)
    test_machines.append(selected)

test_machines = sorted(test_machines)

train_machines = sorted(
    set(machine_modes["machine_id"]) - set(test_machines)
)

print("=" * 80)
print("FINAL STRATIFIED MACHINE SPLIT")
print("=" * 80)

print("\nTEST MACHINES:", len(test_machines))
print(test_machines)

print("\nTRAIN MACHINES:", len(train_machines))
print(train_machines)

print("\nTEST FAILURE-MODE DISTRIBUTION:")
print(
    machine_modes[
        machine_modes["machine_id"].isin(test_machines)
    ]
    .sort_values("failure_mode")
    .to_string(index=False)
)

print("\nTRAIN FAILURE-MODE DISTRIBUTION:")
print(
    machine_modes[
        machine_modes["machine_id"].isin(train_machines)
    ]
    .groupby("failure_mode")
    .size()
    .sort_index()
)

print("\nMachine overlap:")
print(set(train_machines) & set(test_machines))

In [0]:
# ============================================================
# FACTORYMIND RUL V4 — STRATIFIED BASELINE
# ============================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ------------------------------------------------------------
# 1. Load corrected RUL data
# ------------------------------------------------------------

rul_pd = (
    spark.table("workspace.default.factorymind_rul")
    .toPandas()
)

# ------------------------------------------------------------
# 2. Feature selection
# ------------------------------------------------------------

exclude_cols = {
    "machine_id",
    "step",
    "rul",
    "failure_endpoint_step",
    "steps_to_failure",
    "onset_step",
    "failure_mode",
    "health_state",
    "degradation_score",
}

feature_cols = [
    c for c in rul_pd.columns
    if c not in exclude_cols
]

X = rul_pd[feature_cols].copy()
y = rul_pd["rul"].astype(float)

# Numeric features only
X = X.select_dtypes(include=[np.number])

# Replace inf/nan
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())

print("Features:", X.shape[1])
print("Rows:", X.shape[0])

# ------------------------------------------------------------
# 3. Machine-level stratified split
# ------------------------------------------------------------

train_mask = rul_pd["machine_id"].isin(train_machines)
test_mask = rul_pd["machine_id"].isin(test_machines)

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]

y_train = y.loc[train_mask]
y_test = y.loc[test_mask]

print("\nTrain rows:", len(X_train))
print("Test rows :", len(X_test))

print(
    "Train machines:",
    rul_pd.loc[train_mask, "machine_id"].nunique()
)

print(
    "Test machines:",
    rul_pd.loc[test_mask, "machine_id"].nunique()
)

# ------------------------------------------------------------
# 4. Train baseline
# ------------------------------------------------------------

rul_model_v4_stratified = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=3,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rul_model_v4_stratified.fit(
    X_train,
    y_train
)

# ------------------------------------------------------------
# 5. Predictions
# ------------------------------------------------------------

pred = rul_model_v4_stratified.predict(X_test)

# ------------------------------------------------------------
# 6. Overall metrics
# ------------------------------------------------------------

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print("\n" + "=" * 80)
print("FACTORYMIND RUL V4 — STRATIFIED BASELINE")
print("=" * 80)

print(f"MAE  : {mae:.2f} steps")
print(f"RMSE : {rmse:.2f} steps")
print(f"R²   : {r2:.4f}")

print(f"\nMAE  : {mae:.2f} minutes")
print(f"RMSE : {rmse:.2f} minutes")

print(f"\nMAE  : {mae / 60:.2f} hours")
print(f"RMSE : {rmse / 60:.2f} hours")

print(
    "\nCorrelation:",
    np.corrcoef(y_test, pred)[0, 1]
)

In [0]:
# ============================================================
# FAILURE-MODE EVALUATION
# ============================================================

eval_df = pd.DataFrame({
    "machine_id": rul_pd.loc[test_mask, "machine_id"].values,
    "failure_mode": rul_pd.loc[test_mask, "failure_mode"].values,
    "actual_rul": y_test.values,
    "predicted_rul": pred
})

eval_df["error"] = (
    eval_df["predicted_rul"] -
    eval_df["actual_rul"]
)

eval_df["abs_error"] = eval_df["error"].abs()

mode_results = (
    eval_df
    .groupby("failure_mode")
    .agg(
        samples=("actual_rul", "size"),
        actual_mean=("actual_rul", "mean"),
        predicted_mean=("predicted_rul", "mean"),
        MAE=("abs_error", "mean"),
        RMSE=(
            "error",
            lambda x: np.sqrt(np.mean(x ** 2))
        ),
        bias=("error", "mean")
    )
    .round(2)
)

print("=" * 80)
print("RUL PERFORMANCE BY FAILURE MODE")
print("=" * 80)

print(mode_results)

In [0]:
# ============================================================
# RUL HORIZON EVALUATION
# ============================================================

eval_df["rul_range"] = pd.cut(
    eval_df["actual_rul"],
    bins=[-1, 30, 60, 120, 240, 360, 500, 1000],
    labels=[
        "0-30",
        "31-60",
        "61-120",
        "121-240",
        "241-360",
        "361-500",
        "501+"
    ]
)

range_results = (
    eval_df
    .groupby("rul_range", observed=False)
    .agg(
        samples=("actual_rul", "size"),
        actual_mean=("actual_rul", "mean"),
        predicted_mean=("predicted_rul", "mean"),
        MAE=("abs_error", "mean"),
        bias=("error", "mean")
    )
    .round(2)
)

print("=" * 80)
print("RUL PERFORMANCE BY HORIZON")
print("=" * 80)

print(range_results)

In [0]:
# ============================================================
# FACTORYMIND RUL V5
# HISTOGRAM GRADIENT BOOSTING
# ============================================================

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ------------------------------------------------------------
# Remove non-predictive / downstream features
# ------------------------------------------------------------

v5_exclude = {
    "machine_id",
    "step",
    "timestamp",
    "rul",
    "failure_endpoint_step",
    "steps_to_failure",
    "onset_step",
    "failure_mode",
    "health_state",
    "degradation_score",
    "is_anomalous"
}

v5_features = [
    c for c in rul_pd.columns
    if c not in v5_exclude
]

X_v5 = rul_pd[v5_features].copy()

X_v5 = X_v5.select_dtypes(
    include=[np.number]
)

X_v5 = X_v5.replace(
    [np.inf, -np.inf],
    np.nan
)

X_v5 = X_v5.fillna(X_v5.median())

y_v5 = rul_pd["rul"].astype(float)

print("V5 features:", X_v5.shape[1])
print("Rows:", X_v5.shape[0])

# ------------------------------------------------------------
# Same stratified machine split
# ------------------------------------------------------------

X_train_v5 = X_v5.loc[train_mask]
X_test_v5 = X_v5.loc[test_mask]

y_train_v5 = y_v5.loc[train_mask]
y_test_v5 = y_v5.loc[test_mask]

print("Train:", X_train_v5.shape)
print("Test :", X_test_v5.shape)

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

rul_model_v5 = HistGradientBoostingRegressor(
    max_iter=500,
    learning_rate=0.05,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=1.0,
    random_state=42
)

rul_model_v5.fit(
    X_train_v5,
    y_train_v5
)

# ------------------------------------------------------------
# Predict
# ------------------------------------------------------------

pred_v5 = rul_model_v5.predict(X_test_v5)

mae_v5 = mean_absolute_error(
    y_test_v5,
    pred_v5
)

rmse_v5 = np.sqrt(
    mean_squared_error(
        y_test_v5,
        pred_v5
    )
)

r2_v5 = r2_score(
    y_test_v5,
    pred_v5
)

corr_v5 = np.corrcoef(
    y_test_v5,
    pred_v5
)[0, 1]

print("\n" + "=" * 80)
print("FACTORYMIND RUL V5")
print("=" * 80)

print(f"MAE        : {mae_v5:.2f} steps")
print(f"RMSE       : {rmse_v5:.2f} steps")
print(f"R²         : {r2_v5:.4f}")
print(f"Correlation: {corr_v5:.4f}")

In [0]:
# ============================================================
# V5 HORIZON EVALUATION
# ============================================================

eval_v5 = pd.DataFrame({
    "machine_id": rul_pd.loc[test_mask, "machine_id"].values,
    "failure_mode": rul_pd.loc[test_mask, "failure_mode"].values,
    "actual_rul": y_test_v5.values,
    "predicted_rul": pred_v5
})

eval_v5["error"] = (
    eval_v5["predicted_rul"] -
    eval_v5["actual_rul"]
)

eval_v5["abs_error"] = eval_v5["error"].abs()

eval_v5["rul_range"] = pd.cut(
    eval_v5["actual_rul"],
    bins=[-1, 30, 60, 120, 240, 360, 500, 1000],
    labels=[
        "0-30",
        "31-60",
        "61-120",
        "121-240",
        "241-360",
        "361-500",
        "501+"
    ]
)

print("=" * 80)
print("V5 PERFORMANCE BY RUL HORIZON")
print("=" * 80)

print(
    eval_v5
    .groupby("rul_range", observed=False)
    .agg(
        samples=("actual_rul", "size"),
        actual_mean=("actual_rul", "mean"),
        predicted_mean=("predicted_rul", "mean"),
        MAE=("abs_error", "mean"),
        bias=("error", "mean")
    )
    .round(2)
)

In [0]:
print("=" * 80)
print("V5 PERFORMANCE BY FAILURE MODE")
print("=" * 80)

print(
    eval_v5
    .groupby("failure_mode")
    .agg(
        samples=("actual_rul", "size"),
        actual_mean=("actual_rul", "mean"),
        predicted_mean=("predicted_rul", "mean"),
        MAE=("abs_error", "mean"),
        RMSE=(
            "error",
            lambda x: np.sqrt(np.mean(x ** 2))
        ),
        bias=("error", "mean")
    )
    .round(2)
)

In [0]:
# ============================================================
# DIAGNOSTIC: DEGRADATION SCORE vs TRUE RUL
# ============================================================

diag = (
    rul_pd[
        ["degradation_score", "rul"]
    ]
    .copy()
)

print("=" * 80)
print("DEGRADATION SCORE → RUL RELATIONSHIP")
print("=" * 80)

print(
    "Correlation:",
    diag["degradation_score"].corr(diag["rul"])
)

print("\nGrouped relationship:")

diag["degradation_bin"] = pd.cut(
    diag["degradation_score"],
    bins=[
        -0.001,
        0.05,
        0.20,
        0.40,
        0.60,
        0.80,
        1.001
    ],
    labels=[
        "0-0.05",
        "0.05-0.20",
        "0.20-0.40",
        "0.40-0.60",
        "0.60-0.80",
        "0.80-1.00"
    ]
)

print(
    diag
    .groupby("degradation_bin", observed=False)
    .agg(
        samples=("rul", "size"),
        mean_rul=("rul", "mean"),
        median_rul=("rul", "median"),
        min_rul=("rul", "min"),
        max_rul=("rul", "max")
    )
    .round(2)
)

In [0]:
# ============================================================
# FACTORYMIND V6 — LONG-HORIZON TRAJECTORY FEATURES
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

telemetry = (
    spark.table("workspace.default.factorymind_telemetry")
    .filter(F.col("machine_id").isin(failed_machines))
)

sensors = [
    "temperature",
    "vibration",
    "pressure",
    "motor_current",
    "rpm",
    "load",
    "coolant_temperature",
    "coolant_flow",
    "acoustic_level"
]

w = Window.partitionBy("machine_id").orderBy("step")

feature_exprs = []

for sensor in sensors:

    # Rolling statistics
    for window_size in [60, 120]:

        feature_exprs.extend([
            F.avg(F.col(sensor)).over(
                w.rowsBetween(-(window_size - 1), 0)
            ).alias(
                f"{sensor}_mean_{window_size}"
            ),

            F.stddev(F.col(sensor)).over(
                w.rowsBetween(-(window_size - 1), 0)
            ).alias(
                f"{sensor}_std_{window_size}"
            )
        ])

    # Long-term deltas
    for lag in [30, 60, 120]:

        feature_exprs.append(
            (
                F.col(sensor) -
                F.lag(sensor, lag).over(w)
            ).alias(
                f"{sensor}_delta_{lag}"
            )
        )

# Start with original telemetry
v6_features = telemetry.select(
    "machine_id",
    "step",
    *sensors,
    *feature_exprs
)

print("=" * 80)
print("V6 TRAJECTORY FEATURES")
print("=" * 80)

print("Rows:", v6_features.count())
print("Columns:", len(v6_features.columns))

v6_features.printSchema()

In [0]:
# ============================================================
# ADD TRAJECTORY SLOPES
# ============================================================

v6 = v6_features

for sensor in sensors:

    v6 = (
        v6
        .withColumn(
            f"{sensor}_slope_30",
            (
                F.col(sensor) -
                F.lag(sensor, 30).over(w)
            ) / F.lit(30.0)
        )
        .withColumn(
            f"{sensor}_slope_60",
            (
                F.col(sensor) -
                F.lag(sensor, 60).over(w)
            ) / F.lit(60.0)
        )
        .withColumn(
            f"{sensor}_slope_120",
            (
                F.col(sensor) -
                F.lag(sensor, 120).over(w)
            ) / F.lit(120.0)
        )
    )

print("V6 columns:", len(v6.columns))

In [0]:
# ============================================================
# V6 — JOIN CORRECTED RUL TARGET
# ============================================================

from pyspark.sql import functions as F

targets = (
    spark.table("workspace.default.factorymind_rul")
    .select(
        "machine_id",
        "step",
        "rul",
        "failure_mode"
    )
)

v6_rul = (
    v6
    .join(
        targets,
        on=["machine_id", "step"],
        how="inner"
    )
)

print("=" * 80)
print("V6 RUL DATASET")
print("=" * 80)

print("Rows:", v6_rul.count())
print("Columns:", len(v6_rul.columns))

print("\nExpected:")
print("Rows: 17362")
print("Columns: 105")

In [0]:
# Check that the target is correct

v6_rul.select(
    "machine_id",
    "step",
    "rul",
    "failure_mode"
).orderBy(
    "machine_id",
    "step"
).show(20, truncate=False)

In [0]:
# ============================================================
# SAVE V6 DATASET
# ============================================================

V6_TABLE = "workspace.default.factorymind_rul_v6"

(
    v6_rul
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(V6_TABLE)
)

print("Saved:", V6_TABLE)

In [0]:
# ============================================================
# FACTORYMIND RUL V6
# LONG-HORIZON TRAJECTORY FEATURES
# ============================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ------------------------------------------------------------
# Load V6
# ------------------------------------------------------------

v6_pd = (
    spark.table("workspace.default.factorymind_rul_v6")
    .toPandas()
)

print("V6 shape:", v6_pd.shape)

# ------------------------------------------------------------
# Remove target / identifiers / leakage
# ------------------------------------------------------------

exclude_v6 = {
    "machine_id",
    "step",
    "rul",
    "failure_mode",
    "degradation_score",
    "health_state",
    "failure_endpoint_step",
    "steps_to_failure",
    "onset_step",
}

v6_feature_cols = [
    c for c in v6_pd.columns
    if c not in exclude_v6
]

X_v6 = v6_pd[v6_feature_cols].copy()

# Numeric only
X_v6 = X_v6.select_dtypes(
    include=[np.number]
)

# Clean values
X_v6 = X_v6.replace(
    [np.inf, -np.inf],
    np.nan
)

X_v6 = X_v6.fillna(
    X_v6.median()
)

y_v6 = v6_pd["rul"].astype(float)

print("V6 features:", X_v6.shape[1])
print("Rows:", X_v6.shape[0])

# ------------------------------------------------------------
# SAME STRATIFIED SPLIT AS V4
# ------------------------------------------------------------

train_mask_v6 = v6_pd["machine_id"].isin(train_machines)
test_mask_v6 = v6_pd["machine_id"].isin(test_machines)

X_train_v6 = X_v6.loc[train_mask_v6]
X_test_v6 = X_v6.loc[test_mask_v6]

y_train_v6 = y_v6.loc[train_mask_v6]
y_test_v6 = y_v6.loc[test_mask_v6]

print("\nTrain:", X_train_v6.shape)
print("Test :", X_test_v6.shape)

print(
    "Train machines:",
    v6_pd.loc[
        train_mask_v6,
        "machine_id"
    ].nunique()
)

print(
    "Test machines:",
    v6_pd.loc[
        test_mask_v6,
        "machine_id"
    ].nunique()
)

# ------------------------------------------------------------
# RANDOM FOREST
# Same baseline family as V4
# ------------------------------------------------------------

rul_model_v6 = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=3,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rul_model_v6.fit(
    X_train_v6,
    y_train_v6
)

# ------------------------------------------------------------
# PREDICTIONS
# ------------------------------------------------------------

pred_v6 = rul_model_v6.predict(X_test_v6)

# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------

mae_v6 = mean_absolute_error(
    y_test_v6,
    pred_v6
)

rmse_v6 = np.sqrt(
    mean_squared_error(
        y_test_v6,
        pred_v6
    )
)

r2_v6 = r2_score(
    y_test_v6,
    pred_v6
)

corr_v6 = np.corrcoef(
    y_test_v6,
    pred_v6
)[0, 1]

print("\n" + "=" * 80)
print("FACTORYMIND RUL V6")
print("=" * 80)

print(f"MAE        : {mae_v6:.2f} steps")
print(f"RMSE       : {rmse_v6:.2f} steps")
print(f"R²         : {r2_v6:.4f}")
print(f"Correlation: {corr_v6:.4f}")

print(f"\nMAE  : {mae_v6:.2f} minutes")
print(f"RMSE : {rmse_v6:.2f} minutes")

print(f"\nMAE  : {mae_v6/60:.2f} hours")
print(f"RMSE : {rmse_v6/60:.2f} hours")

In [0]:
# ============================================================
# V6 HORIZON EVALUATION
# ============================================================

eval_v6 = pd.DataFrame({
    "machine_id": v6_pd.loc[
        test_mask_v6,
        "machine_id"
    ].values,

    "failure_mode": v6_pd.loc[
        test_mask_v6,
        "failure_mode"
    ].values,

    "actual_rul": y_test_v6.values,

    "predicted_rul": pred_v6
})

eval_v6["error"] = (
    eval_v6["predicted_rul"]
    - eval_v6["actual_rul"]
)

eval_v6["abs_error"] = (
    eval_v6["error"].abs()
)

eval_v6["rul_range"] = pd.cut(
    eval_v6["actual_rul"],
    bins=[
        -1,
        30,
        60,
        120,
        240,
        360,
        500,
        1000
    ],
    labels=[
        "0-30",
        "31-60",
        "61-120",
        "121-240",
        "241-360",
        "361-500",
        "501+"
    ]
)

print("=" * 80)
print("V6 PERFORMANCE BY RUL HORIZON")
print("=" * 80)

print(
    eval_v6
    .groupby(
        "rul_range",
        observed=False
    )
    .agg(
        samples=("actual_rul", "size"),
        actual_mean=("actual_rul", "mean"),
        predicted_mean=("predicted_rul", "mean"),
        MAE=("abs_error", "mean"),
        bias=("error", "mean")
    )
    .round(2)
)

In [0]:
print("=" * 80)
print("V6 PERFORMANCE BY FAILURE MODE")
print("=" * 80)

print(
    eval_v6
    .groupby("failure_mode")
    .agg(
        samples=("actual_rul", "size"),
        actual_mean=("actual_rul", "mean"),
        predicted_mean=("predicted_rul", "mean"),
        MAE=("abs_error", "mean"),
        RMSE=(
            "error",
            lambda x: np.sqrt(np.mean(x ** 2))
        ),
        bias=("error", "mean")
    )
    .round(2)
)

In [0]:
# ============================================================
# V6 PERFORMANCE BY TEST MACHINE
# ============================================================

machine_results_v6 = (
    eval_v6
    .groupby("machine_id")
    .agg(
        samples=("actual_rul", "size"),
        actual_mean=("actual_rul", "mean"),
        predicted_mean=("predicted_rul", "mean"),
        MAE=("abs_error", "mean"),
        RMSE=(
            "error",
            lambda x: np.sqrt(np.mean(x ** 2))
        ),
        bias=("error", "mean")
    )
    .round(2)
)

print("=" * 80)
print("V6 PERFORMANCE BY TEST MACHINE")
print("=" * 80)

print(machine_results_v6)

In [0]:
# ============================================================
# V6 TRAJECTORIES
# ============================================================

# Add step information
eval_v6["step"] = v6_pd.loc[
    test_mask_v6,
    "step"
].values

for machine_id in sorted(
    eval_v6["machine_id"].unique()
):

    m = eval_v6[
        eval_v6["machine_id"] == machine_id
    ].sort_values("step")

    print("\n" + "=" * 60)
    print(machine_id)
    print("=" * 60)

    print(
        m[
            [
                "step",
                "actual_rul",
                "predicted_rul"
            ]
        ]
        .iloc[
            ::max(1, len(m)//10)
        ]
        .round(1)
        .to_string(index=False)
    )

In [0]:
# ============================================================
# V6 — THERMAL OVERLOAD DIAGNOSTIC
# ============================================================

thermal = eval_v6[
    eval_v6["failure_mode"] == "thermal_overload"
].copy()

print("=" * 80)
print("THERMAL OVERLOAD TEST MACHINE")
print("=" * 80)

print(thermal[
    [
        "machine_id",
        "step",
        "actual_rul",
        "predicted_rul",
        "error",
        "abs_error"
    ]
].iloc[::max(1, len(thermal)//15)].round(2).to_string(index=False))

In [0]:
# ============================================================
# THERMAL OVERLOAD — TRAIN VS TEST
# ============================================================

thermal_machines = (
    rul_spark
    .filter(
        F.col("failure_mode") == "thermal_overload"
    )
    .select("machine_id")
    .distinct()
    .toPandas()["machine_id"]
    .tolist()
)

print("Thermal overload machines:")
print(sorted(thermal_machines))

print("\nTest:")
print(
    sorted(
        set(thermal_machines)
        & set(test_machines)
    )
)

print("\nTrain:")
print(
    sorted(
        set(thermal_machines)
        & set(train_machines)
    )
)

In [0]:
# ============================================================
# M-0097 SENSOR TRAJECTORY
# ============================================================

(
    spark.table("workspace.default.factorymind_telemetry")
    .filter(F.col("machine_id") == "M-0097")
    .select(
        "step",
        "temperature",
        "vibration",
        "pressure",
        "motor_current",
        "rpm",
        "load",
        "coolant_temperature",
        "coolant_flow",
        "acoustic_level",
        "degradation_score",
        "health_state"
    )
    .filter(
        (F.col("step") % 50 == 0) |
        (F.col("degradation_score") > 0)
    )
    .orderBy("step")
    .show(100, truncate=False)
)

In [0]:
# ============================================================
# THERMAL OVERLOAD TRAJECTORY COMPARISON
# ============================================================

thermal_machines = ["M-0007", "M-0063", "M-0097"]

thermal_df = (
    spark.table("workspace.default.factorymind_telemetry")
    .filter(F.col("machine_id").isin(thermal_machines))
    .select(
        "machine_id",
        "step",
        "temperature",
        "vibration",
        "pressure",
        "motor_current",
        "rpm",
        "load",
        "coolant_temperature",
        "coolant_flow",
        "acoustic_level",
        "degradation_score",
        "health_state"
    )
    .orderBy("machine_id", "step")
)

display(
    thermal_df
    .filter(F.col("step").isin([0, 100, 200, 300, 400, 500, 600, 700]))
)

In [0]:
# ============================================================
# NORMALIZED TRAJECTORY COMPARISON
# ============================================================

thermal_pd = thermal_df.toPandas()

sensor_cols = [
    "temperature",
    "vibration",
    "pressure",
    "motor_current",
    "rpm",
    "load",
    "coolant_temperature",
    "coolant_flow",
    "acoustic_level",
    "degradation_score"
]

summary = (
    thermal_pd
    .groupby("machine_id")[sensor_cols]
    .agg(["min", "max", "mean", "std"])
)

display(summary)

In [0]:
# ============================================================
# DEGRADATION TRAJECTORY
# ============================================================

display(
    thermal_pd[
        thermal_pd["step"] % 50 == 0
    ][
        [
            "machine_id",
            "step",
            "temperature",
            "vibration",
            "motor_current",
            "load",
            "coolant_temperature",
            "coolant_flow",
            "degradation_score",
            "health_state"
        ]
    ]
    .sort_values(["machine_id", "step"])
)

In [0]:
# ============================================================
# THERMAL OVERLOAD DEGRADATION ONSET
# ============================================================

thermal_pd = (
    spark.table("workspace.default.factorymind_telemetry")
    .filter(F.col("machine_id").isin(["M-0007", "M-0063", "M-0097"]))
    .select(
        "machine_id",
        "step",
        "temperature",
        "vibration",
        "motor_current",
        "load",
        "coolant_temperature",
        "coolant_flow",
        "acoustic_level",
        "degradation_score",
        "health_state"
    )
    .orderBy("machine_id", "step")
    .toPandas()
)

for machine_id in ["M-0007", "M-0063", "M-0097"]:
    
    m = thermal_pd[thermal_pd["machine_id"] == machine_id]
    
    print("=" * 80)
    print(machine_id)
    print("=" * 80)
    
    # First non-zero degradation
    nonzero = m[m["degradation_score"] > 0]
    
    if len(nonzero) > 0:
        print(
            "First degradation:",
            int(nonzero.iloc[0]["step"]),
            "score:",
            nonzero.iloc[0]["degradation_score"]
        )
    
    # First meaningful degradation
    for threshold in [0.05, 0.10, 0.20, 0.40, 0.60, 0.80]:
        rows = m[m["degradation_score"] >= threshold]
        
        if len(rows) > 0:
            print(
                f"First degradation >= {threshold:.2f}:",
                int(rows.iloc[0]["step"])
            )

In [0]:
# ============================================================
# THERMAL FEATURE CORRELATION WITH RUL
# ============================================================

# RUL target
rul_pd = (
    spark.table("workspace.default.factorymind_rul_v6")
    .filter(
        F.col("machine_id").isin(["M-0007", "M-0063", "M-0097"])
    )
    .select(
        "machine_id",
        "step",
        "rul"
    )
)

# Raw telemetry containing degradation_score
telemetry_pd = (
    spark.table("workspace.default.factorymind_telemetry")
    .filter(
        F.col("machine_id").isin(["M-0007", "M-0063", "M-0097"])
    )
    .select(
        "machine_id",
        "step",
        "temperature",
        "vibration",
        "motor_current",
        "load",
        "coolant_temperature",
        "coolant_flow",
        "acoustic_level",
        "degradation_score"
    )
)

# Join target + telemetry
thermal_rul = (
    rul_pd
    .join(
        telemetry_pd,
        on=["machine_id", "step"],
        how="inner"
    )
    .toPandas()
)

print("Rows:", len(thermal_rul))
print("Columns:", thermal_rul.columns.tolist())

thermal_features = [
    "temperature",
    "vibration",
    "motor_current",
    "load",
    "coolant_temperature",
    "coolant_flow",
    "acoustic_level",
    "degradation_score"
]

corr = (
    thermal_rul[thermal_features + ["rul"]]
    .corr()["rul"]
    .sort_values()
)

display(corr.to_frame("correlation_with_rul"))

In [0]:
# ============================================================
# THERMAL FEATURE CORRELATION WITH RUL
# ============================================================

# RUL target
rul_pd = (
    spark.table("workspace.default.factorymind_rul_v6")
    .filter(
        F.col("machine_id").isin(["M-0007", "M-0063", "M-0097"])
    )
    .select(
        "machine_id",
        "step",
        "rul"
    )
)

# Raw telemetry containing degradation_score
telemetry_pd = (
    spark.table("workspace.default.factorymind_telemetry")
    .filter(
        F.col("machine_id").isin(["M-0007", "M-0063", "M-0097"])
    )
    .select(
        "machine_id",
        "step",
        "temperature",
        "vibration",
        "motor_current",
        "load",
        "coolant_temperature",
        "coolant_flow",
        "acoustic_level",
        "degradation_score"
    )
)

# Join target + telemetry
thermal_rul = (
    rul_pd
    .join(
        telemetry_pd,
        on=["machine_id", "step"],
        how="inner"
    )
    .toPandas()
)

print("Rows:", len(thermal_rul))
print("Columns:", thermal_rul.columns.tolist())

thermal_features = [
    "temperature",
    "vibration",
    "motor_current",
    "load",
    "coolant_temperature",
    "coolant_flow",
    "acoustic_level",
    "degradation_score"
]

corr = (
    thermal_rul[thermal_features + ["rul"]]
    .corr()["rul"]
    .sort_values()
)

display(corr.to_frame("correlation_with_rul"))

In [0]:
# ============================================================
# RUL BY DEGRADATION REGIME
# ============================================================

thermal_rul["degradation_bucket"] = pd.cut(
    thermal_rul["degradation_score"],
    bins=[-0.001, 0.05, 0.20, 0.40, 0.60, 0.80, 1.01],
    labels=[
        "0-0.05",
        "0.05-0.20",
        "0.20-0.40",
        "0.40-0.60",
        "0.60-0.80",
        "0.80-1.00"
    ]
)

display(
    thermal_rul
    .groupby(
        ["machine_id", "degradation_bucket"],
        observed=True
    )["rul"]
    .agg(["count", "mean", "min", "max"])
    .reset_index()
)

In [0]:
# ============================================================
# THERMAL DEGRADATION -> RUL BASELINE
# ============================================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Use the already-created thermal_rul dataframe
oracle_df = thermal_rul.copy()

X = oracle_df[["degradation_score"]]
y = oracle_df["rul"]

# Machine-level split: same split as our real experiment
train_machines = ["M-0007", "M-0063"]
test_machines = ["M-0097"]

train_mask = oracle_df["machine_id"].isin(train_machines)
test_mask = oracle_df["machine_id"].isin(test_machines)

oracle_model = RandomForestRegressor(
    n_estimators=500,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

oracle_model.fit(
    X.loc[train_mask],
    y.loc[train_mask]
)

pred = oracle_model.predict(X.loc[test_mask])

actual = y.loc[test_mask].values

mae = mean_absolute_error(actual, pred)
rmse = np.sqrt(mean_squared_error(actual, pred))
r2 = r2_score(actual, pred)

print("=" * 70)
print("THERMAL ORACLE RUL BASELINE")
print("=" * 70)
print(f"MAE  : {mae:.2f} min")
print(f"RMSE : {rmse:.2f} min")
print(f"R²   : {r2:.4f}")

In [0]:
# ============================================================
# ORACLE PERFORMANCE BY DEGRADATION REGIME
# ============================================================

oracle_test = oracle_df.loc[test_mask].copy()
oracle_test["predicted_rul"] = pred

oracle_test["degradation_bucket"] = pd.cut(
    oracle_test["degradation_score"],
    bins=[-0.001, 0.05, 0.20, 0.40, 0.60, 0.80, 1.01],
    labels=[
        "0-0.05",
        "0.05-0.20",
        "0.20-0.40",
        "0.40-0.60",
        "0.60-0.80",
        "0.80-1.00"
    ]
)

oracle_test["abs_error"] = (
    oracle_test["rul"] -
    oracle_test["predicted_rul"]
).abs()

display(
    oracle_test
    .groupby("degradation_bucket", observed=True)
    .agg(
        samples=("rul", "size"),
        actual_mean=("rul", "mean"),
        predicted_mean=("predicted_rul", "mean"),
        MAE=("abs_error", "mean")
    )
    .reset_index()
)

In [0]:
# ============================================================
# V6 FEATURE ABLATION — FEATURE GROUPS
# ============================================================

feature_groups = {
    "raw": [
        c for c in X_train.columns
        if not any(
            token in c
            for token in ["_mean_", "_std_", "_delta_", "_slope_"]
        )
    ],

    "rolling": [
        c for c in X_train.columns
        if "_mean_" in c or "_std_" in c
    ],

    "delta": [
        c for c in X_train.columns
        if "_delta_" in c
    ],

    "slope": [
        c for c in X_train.columns
        if "_slope_" in c
    ]
}

print("=" * 70)
print("V6 FEATURE GROUPS")
print("=" * 70)

for name, cols in feature_groups.items():
    print(f"{name:10s}: {len(cols):3d} features")

In [0]:
# ============================================================
# V6 CUMULATIVE FEATURE ABLATION
# ============================================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np
import pandas as pd

cumulative_groups = {
    "raw_only": feature_groups["raw"],

    "raw_plus_rolling": (
        feature_groups["raw"]
        + feature_groups["rolling"]
    ),

    "raw_plus_rolling_delta": (
        feature_groups["raw"]
        + feature_groups["rolling"]
        + feature_groups["delta"]
    ),

    "all_v6": (
        feature_groups["raw"]
        + feature_groups["rolling"]
        + feature_groups["delta"]
        + feature_groups["slope"]
    )
}

ablation_results = []

for name, cols in cumulative_groups.items():

    cols = list(dict.fromkeys(
        c for c in cols
        if c in X_train.columns
    ))

    print(f"\nRunning: {name}")
    print(f"Features: {len(cols)}")

    model = RandomForestRegressor(
        n_estimators=500,
        min_samples_leaf=3,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train[cols],
        y_train
    )

    pred = model.predict(
        X_test[cols]
    )

    mae = mean_absolute_error(
        y_test,
        pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            pred
        )
    )

    r2 = r2_score(
        y_test,
        pred
    )

    ablation_results.append({
        "feature_set": name,
        "features": len(cols),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

ablation_df = pd.DataFrame(ablation_results)

display(
    ablation_df.sort_values("MAE")
)

In [0]:
# ============================================================
# DEBUG V6 FEATURE COLUMNS
# ============================================================

print("X_train type:", type(X_train))
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nNumber of columns:", len(X_train.columns))

print("\nFirst 100 columns:")
for i, c in enumerate(X_train.columns[:100]):
    print(f"{i:3d}: {c}")

print("\nEmpty / suspicious columns:")
print([
    c for c in X_train.columns
    if not isinstance(c, str) or not c.strip()
])

print("\nColumn name patterns:")
print("mean :", sum("_mean_" in str(c) for c in X_train.columns))
print("std  :", sum("_std_" in str(c) for c in X_train.columns))
print("delta:", sum("_delta_" in str(c) for c in X_train.columns))
print("slope:", sum("_slope_" in str(c) for c in X_train.columns))

In [0]:
# ============================================================
# INSPECT ALL V6 FEATURES
# ============================================================

for i, c in enumerate(X_train.columns):
    print(f"{i:3d}: {c}")

In [0]:
# ============================================================
# SHOW REMAINING V6 FEATURES
# ============================================================

for i, c in enumerate(X_train.columns[20:], start=20):
    print(f"{i:3d}: {c}")

In [0]:
# ============================================================
# V6 FEATURE ABLATION — CORRECTED FOR ACTUAL SCHEMA
# ============================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ------------------------------------------------------------
# 1. Define feature groups from the actual V6 naming scheme
# ------------------------------------------------------------

raw_sensors = [
    "temperature",
    "vibration",
    "pressure",
    "motor_current",
    "rpm",
    "load",
    "coolant_temperature",
    "coolant_flow",
    "acoustic_level"
]

quality_features = [
    c for c in X_train.columns
    if c.endswith("_quality")
]

rolling_features = [
    c for c in X_train.columns
    if "_mean_" in c or "_std_" in c
]

lag_features = [
    c for c in X_train.columns
    if "_lag_" in c
]

anomaly_features = [
    c for c in X_train.columns
    if c == "is_anomalous"
]

feature_groups = {
    "raw": raw_sensors,
    "quality": quality_features,
    "rolling": rolling_features,
    "lag": lag_features,
    "anomaly": anomaly_features
}

print("=" * 70)
print("V6 FEATURE GROUPS")
print("=" * 70)

for name, cols in feature_groups.items():
    valid = [c for c in cols if c in X_train.columns]
    feature_groups[name] = valid
    print(f"{name:10s}: {len(valid):3d}")

print("\nTotal V6 features:", len(X_train.columns))

In [0]:
# ============================================================
# INDIVIDUAL FEATURE-GROUP ABLATION
# ============================================================

individual_results = []

for name, cols in feature_groups.items():

    if len(cols) == 0:
        print(f"Skipping {name}: no features")
        continue

    print(f"\nRunning {name}: {len(cols)} features")

    model = RandomForestRegressor(
        n_estimators=500,
        min_samples_leaf=3,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train[cols],
        y_train
    )

    pred = model.predict(
        X_test[cols]
    )

    individual_results.append({
        "feature_group": name,
        "features": len(cols),
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(
            mean_squared_error(y_test, pred)
        ),
        "R2": r2_score(y_test, pred)
    })

individual_df = pd.DataFrame(individual_results)

display(
    individual_df.sort_values("MAE")
)

In [0]:
# ============================================================
# CUMULATIVE FEATURE ABLATION
# ============================================================

cumulative_groups = {
    "raw_only": (
        feature_groups["raw"]
    ),

    "raw_plus_quality": (
        feature_groups["raw"]
        + feature_groups["quality"]
    ),

    "raw_plus_rolling": (
        feature_groups["raw"]
        + feature_groups["quality"]
        + feature_groups["rolling"]
    ),

    "raw_plus_rolling_lag": (
        feature_groups["raw"]
        + feature_groups["quality"]
        + feature_groups["rolling"]
        + feature_groups["lag"]
    ),

    "all_v6": (
        feature_groups["raw"]
        + feature_groups["quality"]
        + feature_groups["rolling"]
        + feature_groups["lag"]
        + feature_groups["anomaly"]
    )
}

cumulative_results = []

for name, cols in cumulative_groups.items():

    cols = list(dict.fromkeys(
        c for c in cols
        if c in X_train.columns
    ))

    print(f"\nRunning {name}: {len(cols)} features")

    model = RandomForestRegressor(
        n_estimators=500,
        min_samples_leaf=3,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train[cols],
        y_train
    )

    pred = model.predict(
        X_test[cols]
    )

    cumulative_results.append({
        "feature_set": name,
        "features": len(cols),
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(
            mean_squared_error(y_test, pred)
        ),
        "R2": r2_score(y_test, pred)
    })

cumulative_df = pd.DataFrame(cumulative_results)

display(
    cumulative_df.sort_values("MAE")
)

In [0]:
# ============================================================
# PRINT ABLATION RESULTS
# ============================================================

print("\n" + "=" * 80)
print("INDIVIDUAL FEATURE-GROUP ABLATION")
print("=" * 80)

print(
    individual_df
    .sort_values("MAE")
    .to_string(index=False)
)

print("\n" + "=" * 80)
print("CUMULATIVE FEATURE ABLATION")
print("=" * 80)

print(
    cumulative_df
    .sort_values("MAE")
    .to_string(index=False)
)

In [0]:
# ============================================================
# FIND V6 FEATURE DATAFRAME / SCHEMA
# ============================================================

print("Variables containing 'v6':")

for name in sorted(globals().keys()):
    if "v6" in name.lower():
        try:
            obj = globals()[name]
            if hasattr(obj, "shape") and hasattr(obj, "columns"):
                print(
                    f"{name}: "
                    f"shape={obj.shape}, "
                    f"columns={len(obj.columns)}"
                )
        except Exception:
            pass

In [0]:
# ============================================================
# INSPECT SAVED V6 DELTA TABLE
# ============================================================

v6_spark = spark.table(
    "workspace.default.factorymind_rul_v6"
)

print("Rows:", v6_spark.count())
print("Columns:", len(v6_spark.columns))

for i, c in enumerate(v6_spark.columns):
    print(f"{i:3d}: {c}")

In [0]:
# ============================================================
# CORRECT V6 FEATURE GROUPS
# ============================================================

v6_columns = list(X_train_v6.columns)

raw_sensors = [
    "temperature",
    "vibration",
    "pressure",
    "motor_current",
    "rpm",
    "load",
    "coolant_temperature",
    "coolant_flow",
    "acoustic_level"
]

rolling_features = [
    c for c in v6_columns
    if "_mean_" in c or "_std_" in c
]

delta_features = [
    c for c in v6_columns
    if "_delta_" in c
]

slope_features = [
    c for c in v6_columns
    if "_slope_" in c
]

feature_groups_v6 = {
    "raw": [
        c for c in raw_sensors
        if c in v6_columns
    ],
    "rolling": rolling_features,
    "delta": delta_features,
    "slope": slope_features
}

print("=" * 70)
print("CORRECT V6 FEATURE GROUPS")
print("=" * 70)

for name, cols in feature_groups_v6.items():
    print(f"{name:10s}: {len(cols):3d}")

print("\nTotal:", sum(len(v) for v in feature_groups_v6.values()))

In [0]:
# ============================================================
# REAL V6 FEATURE ABLATION
# ============================================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np
import pandas as pd

results_v6_ablation = []

for name, cols in feature_groups_v6.items():

    if not cols:
        continue

    print(f"Running {name}: {len(cols)} features")

    model = RandomForestRegressor(
        n_estimators=500,
        min_samples_leaf=3,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_v6[cols],
        y_train
    )

    pred = model.predict(
        X_test_v6[cols]
    )

    results_v6_ablation.append({
        "feature_group": name,
        "features": len(cols),
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(
            mean_squared_error(y_test, pred)
        ),
        "R2": r2_score(y_test, pred)
    })

display(
    pd.DataFrame(results_v6_ablation)
    .sort_values("MAE")
)

In [0]:
# ============================================================
# REAL V6 CUMULATIVE ABLATION
# ============================================================

cumulative_v6 = {
    "raw": (
        feature_groups_v6["raw"]
    ),

    "raw_plus_rolling": (
        feature_groups_v6["raw"]
        + feature_groups_v6["rolling"]
    ),

    "raw_plus_rolling_delta": (
        feature_groups_v6["raw"]
        + feature_groups_v6["rolling"]
        + feature_groups_v6["delta"]
    ),

    "all_v6": (
        feature_groups_v6["raw"]
        + feature_groups_v6["rolling"]
        + feature_groups_v6["delta"]
        + feature_groups_v6["slope"]
    )
}

results_cumulative_v6 = []

for name, cols in cumulative_v6.items():

    cols = list(dict.fromkeys(
        c for c in cols
        if c in X_train_v6.columns
    ))

    print(f"Running {name}: {len(cols)} features")

    model = RandomForestRegressor(
        n_estimators=500,
        min_samples_leaf=3,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_v6[cols],
        y_train
    )

    pred = model.predict(
        X_test_v6[cols]
    )

    results_cumulative_v6.append({
        "feature_set": name,
        "features": len(cols),
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(
            mean_squared_error(y_test, pred)
        ),
        "R2": r2_score(y_test, pred)
    })

display(
    pd.DataFrame(results_cumulative_v6)
    .sort_values("MAE")
)

In [0]:
# ============================================================
# PRINT CORRECT V6 ABLATION RESULTS
# ============================================================

individual_v6_df = (
    pd.DataFrame(results_v6_ablation)
    .sort_values("MAE")
)

cumulative_v6_df = (
    pd.DataFrame(results_cumulative_v6)
    .sort_values("MAE")
)

print("=" * 80)
print("INDIVIDUAL V6 FEATURE-GROUP ABLATION")
print("=" * 80)
print(individual_v6_df.to_string(index=False))

print("\n" + "=" * 80)
print("CUMULATIVE V6 FEATURE ABLATION")
print("=" * 80)
print(cumulative_v6_df.to_string(index=False))

In [0]:
# ============================================================
# V6-DYNAMIC — WINNING FEATURE BASELINE
# ============================================================

dynamic_features = list(dict.fromkeys(
    feature_groups_v6["delta"]
    + feature_groups_v6["slope"]
))

print("Dynamic features:", len(dynamic_features))

dynamic_model = RandomForestRegressor(
    n_estimators=500,
    min_samples_leaf=3,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

dynamic_model.fit(
    X_train_v6[dynamic_features],
    y_train
)

dynamic_pred = dynamic_model.predict(
    X_test_v6[dynamic_features]
)

dynamic_mae = mean_absolute_error(
    y_test,
    dynamic_pred
)

dynamic_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        dynamic_pred
    )
)

dynamic_r2 = r2_score(
    y_test,
    dynamic_pred
)

print("=" * 70)
print("V6-DYNAMIC BASELINE")
print("=" * 70)
print(f"Features: {len(dynamic_features)}")
print(f"MAE     : {dynamic_mae:.2f} min")
print(f"RMSE    : {dynamic_rmse:.2f} min")
print(f"R²      : {dynamic_r2:.4f}")

In [0]:
# ============================================================
# CHECK V6 TEST OBJECTS
# ============================================================

print("X_train_v6:", X_train_v6.shape)
print("X_test_v6 :", X_test_v6.shape)
print("X_v6      :", X_v6.shape)

print("y_train   :", len(y_train))
print("y_test    :", len(y_test))

print("\nTest machines:")
print(test_machines)

print("\nAvailable V6-related objects:")
for name in [
    "X_train_v6",
    "X_test_v6",
    "X_v6",
    "y_train",
    "y_test",
    "eval_v6",
    "machine_results_v6",
    "v6_pd"
]:
    if name in globals():
        obj = globals()[name]
        print(
            f"{name:20s}",
            type(obj),
            getattr(obj, "shape", None)
        )

In [0]:
print("eval_v6 shape:", eval_v6.shape)
print("eval_v6 columns:")
print(eval_v6.columns.tolist())

display(eval_v6.head(10))

In [0]:
print("eval_v6 shape:", eval_v6.shape)
print("\nColumns:")
print(eval_v6.columns.tolist())

display(eval_v6.head(20))

In [0]:
# ============================================================
# V6-DYNAMIC — HORIZON + FAILURE-MODE DIAGNOSTICS
# ============================================================

import pandas as pd
import numpy as np

# Use the existing V6 evaluation metadata
dynamic_eval = eval_v6.copy()

# Replace predictions with V6-Dynamic predictions
dynamic_eval["predicted_rul"] = dynamic_pred

dynamic_eval["error"] = (
    dynamic_eval["predicted_rul"] - dynamic_eval["actual_rul"]
)

dynamic_eval["abs_error"] = dynamic_eval["error"].abs()

# ------------------------------------------------------------
# 1. Overall metrics
# ------------------------------------------------------------

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(
    dynamic_eval["actual_rul"],
    dynamic_eval["predicted_rul"]
)

rmse = np.sqrt(
    mean_squared_error(
        dynamic_eval["actual_rul"],
        dynamic_eval["predicted_rul"]
    )
)

r2 = r2_score(
    dynamic_eval["actual_rul"],
    dynamic_eval["predicted_rul"]
)

corr = dynamic_eval[
    ["actual_rul", "predicted_rul"]
].corr().iloc[0, 1]

print("=" * 70)
print("V6-DYNAMIC — OVERALL PERFORMANCE")
print("=" * 70)
print(f"MAE  : {mae:.2f} min")
print(f"RMSE : {rmse:.2f} min")
print(f"R²   : {r2:.4f}")
print(f"Corr : {corr:.4f}")


# ------------------------------------------------------------
# 2. RUL horizon
# ------------------------------------------------------------

def rul_bucket(rul):
    if rul <= 30:
        return "0-30"
    elif rul <= 60:
        return "31-60"
    elif rul <= 120:
        return "61-120"
    elif rul <= 240:
        return "121-240"
    elif rul <= 360:
        return "241-360"
    elif rul <= 500:
        return "361-500"
    else:
        return "501+"

dynamic_eval["rul_range"] = (
    dynamic_eval["actual_rul"].apply(rul_bucket)
)

bucket_order = [
    "0-30",
    "31-60",
    "61-120",
    "121-240",
    "241-360",
    "361-500",
    "501+"
]

horizon_results = (
    dynamic_eval
    .groupby("rul_range", observed=False)
    .agg(
        samples=("actual_rul", "size"),
        actual_mean=("actual_rul", "mean"),
        predicted_mean=("predicted_rul", "mean"),
        MAE=("abs_error", "mean"),
        bias=("error", "mean")
    )
    .reindex(bucket_order)
)

print("\n" + "=" * 70)
print("V6-DYNAMIC — PERFORMANCE BY RUL HORIZON")
print("=" * 70)

display(horizon_results.round(2))


# ------------------------------------------------------------
# 3. Failure-mode performance
# ------------------------------------------------------------

mode_results = (
    dynamic_eval
    .groupby("failure_mode")
    .apply(
        lambda g: pd.Series({
            "samples": len(g),
            "actual_mean": g["actual_rul"].mean(),
            "predicted_mean": g["predicted_rul"].mean(),
            "MAE": g["abs_error"].mean(),
            "RMSE": np.sqrt(np.mean(g["error"] ** 2)),
            "bias": g["error"].mean()
        }),
        include_groups=False
    )
    .sort_values("RMSE", ascending=False)
)

print("\n" + "=" * 70)
print("V6-DYNAMIC — PERFORMANCE BY FAILURE MODE")
print("=" * 70)

display(mode_results.round(2))


# ------------------------------------------------------------
# 4. Worst individual predictions
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("V6-DYNAMIC — 20 WORST PREDICTIONS")
print("=" * 70)

display(
    dynamic_eval[
        [
            "machine_id",
            "failure_mode",
            "step",
            "actual_rul",
            "predicted_rul",
            "error",
            "abs_error",
            "rul_range"
        ]
    ]
    .sort_values("abs_error", ascending=False)
    .head(20)
    .round(2)
)

In [0]:
print("=" * 70)
print("V6-DYNAMIC — HORIZON RESULTS")
print("=" * 70)

display(horizon_results.round(2))

print("=" * 70)
print("V6-DYNAMIC — FAILURE-MODE RESULTS")
print("=" * 70)

display(mode_results.round(2))

In [0]:
print("\n" + "=" * 70)
print("V6-DYNAMIC — HORIZON RESULTS")
print("=" * 70)
print(horizon_results.round(2).to_string())

print("\n" + "=" * 70)
print("V6-DYNAMIC — FAILURE-MODE RESULTS")
print("=" * 70)
print(mode_results.round(2).to_string())

In [0]:
# ============================================================
# V7 DIAGNOSTIC — DELTA vs SLOPE FEATURE INDEPENDENCE
# ============================================================

delta_features = [
    c for c in X_v6.columns
    if "_delta_" in c
]

slope_features = [
    c for c in X_v6.columns
    if "_slope_" in c
]

print("Delta features:", len(delta_features))
print("Slope features:", len(slope_features))

print("\nFirst 10 delta features:")
print(delta_features[:10])

print("\nFirst 10 slope features:")
print(slope_features[:10])

# Compare corresponding delta/slope columns
pairs = []

for d in delta_features:
    sensor = d.split("_delta_")[0]
    window = d.split("_delta_")[1]

    s = f"{sensor}_slope_{window}"

    if s in X_v6.columns:
        pairs.append((d, s))

print("\nMatched pairs:", len(pairs))

# Numerical comparison
results = []

for d, s in pairs:
    d_vals = X_v6[d].values
    s_vals = X_v6[s].values

    results.append({
        "delta": d,
        "slope": s,
        "mean_abs_difference": np.mean(np.abs(d_vals - s_vals)),
        "max_abs_difference": np.max(np.abs(d_vals - s_vals)),
        "correlation": np.corrcoef(d_vals, s_vals)[0, 1]
    })

delta_slope_check = pd.DataFrame(results)

display(delta_slope_check.round(6))

In [0]:
# ============================================================
# V7 — TEMPORAL WINDOW ABLATION
# 30 vs 60 vs 120 STEP DYNAMICS
# ============================================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Feature groups
# ------------------------------------------------------------

delta_30 = [
    c for c in X_v6.columns
    if "_delta_30" in c
]

delta_60 = [
    c for c in X_v6.columns
    if "_delta_60" in c
]

delta_120 = [
    c for c in X_v6.columns
    if "_delta_120" in c
]

print("30-step features :", len(delta_30))
print("60-step features :", len(delta_60))
print("120-step features:", len(delta_120))


# ------------------------------------------------------------
# Train/evaluate helper
# ------------------------------------------------------------

def evaluate_v7(features, name):

    model = RandomForestRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=3,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_v6[features],
        y_train
    )

    pred = model.predict(
        X_test_v6[features]
    )

    mae = mean_absolute_error(y_test, pred)

    rmse = np.sqrt(
        mean_squared_error(y_test, pred)
    )

    r2 = r2_score(y_test, pred)

    corr = np.corrcoef(
        y_test,
        pred
    )[0, 1]

    return {
        "model": name,
        "features": len(features),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Correlation": corr,
        "pred": pred,
        "model_object": model
    }


# ------------------------------------------------------------
# Run experiments
# ------------------------------------------------------------

v7_30 = evaluate_v7(
    delta_30,
    "30-step dynamics"
)

v7_60 = evaluate_v7(
    delta_60,
    "60-step dynamics"
)

v7_120 = evaluate_v7(
    delta_120,
    "120-step dynamics"
)


# ------------------------------------------------------------
# Comparison table
# ------------------------------------------------------------

v7_results = pd.DataFrame([
    {
        "model": v7_30["model"],
        "features": v7_30["features"],
        "MAE": v7_30["MAE"],
        "RMSE": v7_30["RMSE"],
        "R2": v7_30["R2"],
        "Correlation": v7_30["Correlation"]
    },
    {
        "model": v7_60["model"],
        "features": v7_60["features"],
        "MAE": v7_60["MAE"],
        "RMSE": v7_60["RMSE"],
        "R2": v7_60["R2"],
        "Correlation": v7_60["Correlation"]
    },
    {
        "model": v7_120["model"],
        "features": v7_120["features"],
        "MAE": v7_120["MAE"],
        "RMSE": v7_120["RMSE"],
        "R2": v7_120["R2"],
        "Correlation": v7_120["Correlation"]
    }
])

display(v7_results.round(4))

In [0]:
print(
    v7_results[
        ["model", "features", "MAE", "RMSE", "R2", "Correlation"]
    ].round(4).to_string(index=False)
)

In [0]:
# ============================================================
# V8 — MULTI-SCALE TEMPORAL DYNAMICS
# 30 + 60 + 120 STEP DELTAS
# ============================================================

multi_scale_features = (
    delta_30 +
    delta_60 +
    delta_120
)

print("Multi-scale features:", len(multi_scale_features))


v8_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=3,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

v8_model.fit(
    X_train_v6[multi_scale_features],
    y_train
)

v8_pred = v8_model.predict(
    X_test_v6[multi_scale_features]
)


# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

v8_mae = mean_absolute_error(
    y_test,
    v8_pred
)

v8_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        v8_pred
    )
)

v8_r2 = r2_score(
    y_test,
    v8_pred
)

v8_corr = np.corrcoef(
    y_test,
    v8_pred
)[0, 1]


print("=" * 60)
print("V8 — MULTI-SCALE DYNAMICS")
print("=" * 60)

print(f"Features    : {len(multi_scale_features)}")
print(f"MAE         : {v8_mae:.2f} min")
print(f"RMSE        : {v8_rmse:.2f} min")
print(f"R²          : {v8_r2:.4f}")
print(f"Correlation : {v8_corr:.4f}")

In [0]:
comparison = pd.DataFrame([
    {
        "model": "V6-Dynamic",
        "features": 54,
        "MAE": 69.96,
        "RMSE": 96.05,
        "R2": 0.7390,
        "Correlation": 0.8859
    },
    {
        "model": "V7-30",
        "features": 9,
        "MAE": v7_30["MAE"],
        "RMSE": v7_30["RMSE"],
        "R2": v7_30["R2"],
        "Correlation": v7_30["Correlation"]
    },
    {
        "model": "V7-60",
        "features": 9,
        "MAE": v7_60["MAE"],
        "RMSE": v7_60["RMSE"],
        "R2": v7_60["R2"],
        "Correlation": v7_60["Correlation"]
    },
    {
        "model": "V7-120",
        "features": 9,
        "MAE": v7_120["MAE"],
        "RMSE": v7_120["RMSE"],
        "R2": v7_120["R2"],
        "Correlation": v7_120["Correlation"]
    },
    {
        "model": "V8-MultiScale",
        "features": len(multi_scale_features),
        "MAE": v8_mae,
        "RMSE": v8_rmse,
        "R2": v8_r2,
        "Correlation": v8_corr
    }
])

display(comparison.round(4))

In [0]:

# ============================================================
# FACTORYMIND — V8 MULTI-SCALE RUL MODEL
# ============================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Combine the three temporal windows
multi_scale_features = list(dict.fromkeys(
    delta_30 + delta_60 + delta_120
))

print("Number of features:", len(multi_scale_features))

# Train the model using the existing machine-level split
v8_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=3,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

v8_model.fit(
    X_train_v6[multi_scale_features],
    y_train
)

# Predict on the held-out test machines
v8_pred = v8_model.predict(
    X_test_v6[multi_scale_features]
)

# Evaluate
v8_mae = mean_absolute_error(y_test, v8_pred)
v8_rmse = np.sqrt(mean_squared_error(y_test, v8_pred))
v8_r2 = r2_score(y_test, v8_pred)
v8_corr = np.corrcoef(y_test, v8_pred)[0, 1]

print("\n" + "=" * 55)
print("FACTORYMIND — V8 MULTI-SCALE RESULTS")
print("=" * 55)
print(f"Features    : {len(multi_scale_features)}")
print(f"MAE         : {v8_mae:.2f} minutes")
print(f"RMSE        : {v8_rmse:.2f} minutes")
print(f"R²          : {v8_r2:.4f}")
print(f"Correlation : {v8_corr:.4f}")


In [0]:

comparison = pd.DataFrame([
    {
        "model": "V6-Dynamic",
        "features": 54,
        "MAE": 69.96,
        "RMSE": 96.05,
        "R2": 0.7390,
        "Correlation": 0.8859
    },
    {
        "model": "V7-30",
        "features": len(delta_30),
        "MAE": v7_30["MAE"],
        "RMSE": v7_30["RMSE"],
        "R2": v7_30["R2"],
        "Correlation": v7_30["Correlation"]
    },
    {
        "model": "V7-60",
        "features": len(delta_60),
        "MAE": v7_60["MAE"],
        "RMSE": v7_60["RMSE"],
        "R2": v7_60["R2"],
        "Correlation": v7_60["Correlation"]
    },
    {
        "model": "V7-120",
        "features": len(delta_120),
        "MAE": v7_120["MAE"],
        "RMSE": v7_120["RMSE"],
        "R2": v7_120["R2"],
        "Correlation": v7_120["Correlation"]
    },
    {
        "model": "V8-MultiScale",
        "features": len(multi_scale_features),
        "MAE": v8_mae,
        "RMSE": v8_rmse,
        "R2": v8_r2,
        "Correlation": v8_corr
    }
])

display(
    comparison.sort_values("MAE").round(4)
)


In [0]:

comparison = pd.DataFrame([
    {
        "model": "V6-Dynamic",
        "features": 54,
        "MAE": 69.96,
        "RMSE": 96.05,
        "R2": 0.7390,
        "Correlation": 0.8859
    },
    {
        "model": "V7-30",
        "features": len(delta_30),
        "MAE": v7_30["MAE"],
        "RMSE": v7_30["RMSE"],
        "R2": v7_30["R2"],
        "Correlation": v7_30["Correlation"]
    },
    {
        "model": "V7-60",
        "features": len(delta_60),
        "MAE": v7_60["MAE"],
        "RMSE": v7_60["RMSE"],
        "R2": v7_60["R2"],
        "Correlation": v7_60["Correlation"]
    },
    {
        "model": "V7-120",
        "features": len(delta_120),
        "MAE": v7_120["MAE"],
        "RMSE": v7_120["RMSE"],
        "R2": v7_120["R2"],
        "Correlation": v7_120["Correlation"]
    },
    {
        "model": "V8-MultiScale",
        "features": len(multi_scale_features),
        "MAE": v8_mae,
        "RMSE": v8_rmse,
        "R2": v8_r2,
        "Correlation": v8_corr
    }
])

display(
    comparison.sort_values("MAE").round(4)
)


In [0]:

import os

print("Current directory:", os.getcwd())
print("Workspace path:", "/Workspace/Users/praveenbabugeddada@gmail.com/factorymind")
print("Directory exists:", os.path.isdir(
    "/Workspace/Users/praveenbabugeddada@gmail.com/factorymind"
))


In [0]:

from pathlib import Path
import json
import joblib

artifact_dir = Path("/tmp/factorymind/artifacts/rul_v6_dynamic")
artifact_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(dynamic_model, artifact_dir / "model.joblib")

with open(artifact_dir / "features.json", "w") as f:
    json.dump(dynamic_features, f, indent=2)

print("Saved files:")
for file in artifact_dir.iterdir():
    print(file.name, file.stat().st_size, "bytes")


In [0]:

# ============================================================
# FACTORYMIND — SAVE V6-DYNAMIC MODEL
# Temporary storage workaround
# ============================================================

import json
import joblib
from pathlib import Path

assert "dynamic_model" in globals(), "dynamic_model not found"
assert "dynamic_features" in globals(), "dynamic_features not found"

# Use writable temporary storage instead of protected Workspace files
artifact_dir = Path("/tmp/factorymind/artifacts/rul_v6_dynamic")
artifact_dir.mkdir(parents=True, exist_ok=True)

model_path = artifact_dir / "model.joblib"
features_path = artifact_dir / "features.json"
metadata_path = artifact_dir / "metadata.json"

# Save trained model
joblib.dump(dynamic_model, model_path)

# Save exact feature list
with open(features_path, "w") as f:
    json.dump(list(dynamic_features), f, indent=2)

# Save evaluation metadata
metadata = {
    "model_name": "FactoryMind RUL V6-Dynamic",
    "algorithm": "RandomForestRegressor",
    "feature_count": len(dynamic_features),
    "metrics": {
        "mae_minutes": 69.96,
        "rmse_minutes": 96.05,
        "r2": 0.7390,
        "correlation": 0.8859
    },
    "dataset": "workspace.default.factorymind_rul_v6",
    "evaluation": "held-out machine-level test split",
    "status": "experimental_baseline"
}

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

# Verify files
assert model_path.exists()
assert features_path.exists()
assert metadata_path.exists()

print("Model artifacts saved successfully:")
for path in [model_path, features_path, metadata_path]:
    print(f"{path} — {path.stat().st_size:,} bytes")

print("Feature count:", len(dynamic_features))


In [0]:

# ============================================================
# FACTORYMIND — LOAD V6-DYNAMIC MODEL ARTIFACTS
# ============================================================

from pathlib import Path
import json
import joblib

# Match the temporary directory used by the corrected save cell
ARTIFACT_DIR = Path(
    "/tmp/factorymind/artifacts/rul_v6_dynamic"
)

MODEL_PATH = ARTIFACT_DIR / "model.joblib"
FEATURES_PATH = ARTIFACT_DIR / "features.json"
METADATA_PATH = ARTIFACT_DIR / "metadata.json"

print("Checking artifact files...")

for path in [MODEL_PATH, FEATURES_PATH, METADATA_PATH]:
    print(f"{path.name}: {'FOUND' if path.exists() else 'MISSING'}")

if not all(
    p.exists()
    for p in [MODEL_PATH, FEATURES_PATH, METADATA_PATH]
):
    print("\nArtifacts are missing from temporary storage.")
    print("Run the corrected SAVE V6-DYNAMIC MODEL cell first.")
else:
    rul_model = joblib.load(MODEL_PATH)

    with open(FEATURES_PATH) as f:
        rul_features = json.load(f)

    with open(METADATA_PATH) as f:
        rul_metadata = json.load(f)

    print("\nModel:", rul_metadata["model_name"])
    print("Features:", len(rul_features))
    print(
        "Expected features:",
        getattr(rul_model, "n_features_in_", "unknown")
    )

    assert len(rul_features) == rul_model.n_features_in_, (
        "Saved feature list does not match the model."
    )

    print("\nModel loaded and feature count verified.")


In [0]:

from pathlib import Path

for base in [
    Path("/tmp/factorymind"),
    Path("/Workspace/Users/praveenbabugeddada@gmail.com/factorymind"),
]:
    print(f"\nDirectory: {base}")
    print("Exists:", base.exists())

    if base.exists():
        for path in list(base.rglob("*"))[:20]:
            print(path)


In [0]:

# FACTORYMIND — INITIALIZE RUL INFERENCE

import numpy as np
import pandas as pd
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

assert "dynamic_model" in globals(), (
    "dynamic_model is missing. Rerun the V6-Dynamic training cell."
)
assert "dynamic_features" in globals(), (
    "dynamic_features is missing. Rerun the V6-Dynamic feature cell."
)
assert "X_test_v6" in globals(), "X_test_v6 is missing."
assert "y_test" in globals(), "y_test is missing."

rul_model = dynamic_model
rul_features = list(dynamic_features)

missing = [c for c in rul_features if c not in X_test_v6.columns]
assert not missing, f"Missing model features: {missing}"

X_rul_test = X_test_v6[rul_features].copy()

assert X_rul_test.shape[1] == rul_model.n_features_in_, (
    f"Expected {rul_model.n_features_in_} features, "
    f"got {X_rul_test.shape[1]}"
)

print("RUL model ready.")
print("Features:", len(rul_features))
print("Test matrix:", X_rul_test.shape)


In [0]:

# FACTORYMIND — RUL INFERENCE EVALUATION

rul_predictions = rul_model.predict(X_rul_test)

y_true = np.asarray(y_test).reshape(-1)

assert len(y_true) == len(rul_predictions), (
    "Actual and predicted RUL lengths differ."
)

rul_evaluation = pd.DataFrame({
    "actual_rul": y_true,
    "predicted_rul": rul_predictions
})

rul_evaluation["error"] = (
    rul_evaluation["predicted_rul"]
    - rul_evaluation["actual_rul"]
)

rul_evaluation["abs_error"] = rul_evaluation["error"].abs()

print("MAE :", round(mean_absolute_error(y_true, rul_predictions), 2))
print("RMSE:", round(np.sqrt(mean_squared_error(y_true, rul_predictions)), 2))
print("R²  :", round(r2_score(y_true, rul_predictions), 4))
print("Correlation:", round(np.corrcoef(y_true, rul_predictions)[0, 1], 4))

display(rul_evaluation.head(10))


In [0]:

# FACTORYMIND — MACHINE-LEVEL RUL PREDICTIONS

assert "eval_v6" in globals(), "eval_v6 is missing."

print("Evaluation rows:", len(eval_v6))
print("Prediction rows:", len(rul_predictions))
print("Test feature rows:", len(X_test_v6))

assert len(eval_v6) == len(rul_predictions) == len(X_test_v6), (
    "Row counts differ. Do not combine the frames yet."
)

required_columns = ["machine_id", "step", "failure_mode"]
missing = [c for c in required_columns if c not in eval_v6.columns]

assert not missing, f"Missing columns in eval_v6: {missing}"

machine_predictions = eval_v6[
    required_columns
].copy().reset_index(drop=True)

machine_predictions["actual_rul"] = np.asarray(y_test)
machine_predictions["predicted_rul"] = rul_predictions
machine_predictions["prediction_error"] = (
    machine_predictions["predicted_rul"]
    - machine_predictions["actual_rul"]
)
machine_predictions["absolute_error"] = (
    machine_predictions["prediction_error"].abs()
)
machine_predictions["model_version"] = "v6_dynamic"

display(machine_predictions.head(10))


In [0]:

# FACTORYMIND — VERIFY TEST SET ALIGNMENT

print("eval_v6 columns:", eval_v6.columns.tolist())
print("X_test_v6 index type:", type(X_test_v6.index).__name__)
print("eval_v6 index type:", type(eval_v6.index).__name__)

# Compare indices if both objects preserve their original indices
if X_test_v6.index.equals(eval_v6.index):
    print("PASS: X_test_v6 and eval_v6 indices match.")
else:
    print("WARNING: indices differ; row alignment is not confirmed.")

print("\nFirst 5 test indices:")
print(X_test_v6.index[:5].tolist())

print("\nFirst 5 evaluation indices:")
print(eval_v6.index[:5].tolist())

print("\nFirst 5 actual RUL values:")
print(np.asarray(y_test)[:5])

print("\nFirst 5 predicted RUL values:")
print(np.asarray(rul_predictions)[:5])


In [0]:

# FACTORYMIND — SAVE VERIFIED RUL PREDICTIONS

output_table = "workspace.default.factorymind_rul_predictions"

spark_output = spark.createDataFrame(machine_predictions)

(
    spark_output.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(output_table)
)

print("Saved Delta table:", output_table)

display(
    spark.table(output_table).limit(10)
)


In [0]:

# FACTORYMIND — DISCOVER ANOMALY DETECTOR

import inspect

candidate_names = [
    "isolation_forest",
    "anomaly_model",
    "iso_forest",
    "iforest",
    "anomaly_detector",
    "model",
    "X_test",
    "X_train",
    "y_pred",
    "anomaly_predictions",
    "predictions",
    "y_pred_if",
    "if_predictions",
]

for name in candidate_names:
    if name in globals():
        obj = globals()[name]
        print(
            f"{name}: type={type(obj).__name__}, "
            f"shape={getattr(obj, 'shape', 'N/A')}"
        )
        if hasattr(obj, "n_features_in_"):
            print("  Expected features:", obj.n_features_in_)
        if hasattr(obj, "feature_names_in_"):
            print("  Feature names:", list(obj.feature_names_in_)[:10])


In [0]:

# FACTORYMIND — ANOMALY FEATURE SCHEMA DIAGNOSTIC

print("Anomaly model:", type(iso_forest).__name__)
print("Expected feature count:", iso_forest.n_features_in_)

if hasattr(iso_forest, "feature_names_in_"):
    anomaly_features = list(iso_forest.feature_names_in_)
    print("\nAnomaly feature names:")
    for i, feature in enumerate(anomaly_features, start=1):
        print(f"{i:02d}. {feature}")
else:
    anomaly_features = None
    print("Model does not expose feature_names_in_.")

print("\nCandidate feature DataFrames:")
for name in [
    "X_train", "X_test", "X_train_v6", "X_test_v6",
    "X", "features", "feature_df", "df_features"
]:
    if name in globals():
        obj = globals()[name]
        if isinstance(obj, pd.DataFrame):
            print(
                f"{name}: shape={obj.shape}, "
                f"columns={len(obj.columns)}"
            )


In [0]:

# FACTORYMIND — RECOVER ANOMALY TRAINING MATRIX
import pandas as pd

expected = list(iso_forest.feature_names_in_)

print("Expected features:", len(expected))
print("Duplicate names expected by model:")
print(pd.Index(expected)[pd.Index(expected).duplicated(keep=False)].tolist())

print("\nDataFrames in notebook matching anomaly schema:")
for name, obj in list(globals().items()):
    if not isinstance(obj, pd.DataFrame):
        continue

    cols = list(obj.columns)
    overlap = sum(c in cols for c in expected)
    exact_order = cols == expected
    exact_multiset = sorted(map(str, cols)) == sorted(map(str, expected))

    if len(cols) in (81, 82, 83) or overlap >= 75:
        duplicates = pd.Index(cols)[pd.Index(cols).duplicated(keep=False)].tolist()
        print(
            f"{name}: shape={obj.shape}, overlap={overlap}/82, "
            f"exact_order={exact_order}, "
            f"same_columns_including_duplicates={exact_multiset}, "
            f"duplicates={duplicates[:10]}"
        )


In [0]:

# FACTORYMIND — VERIFY ORIGINAL ANOMALY FEATURE ORDER
from collections import Counter

expected = list(iso_forest.feature_names_in_)

for name in ["pdf", "healthy_train", "rul_pdf", "train_rul", "test_rul"]:
    if name not in globals():
        continue

    df = globals()[name]
    actual = list(df.columns)

    print(f"\n{'=' * 60}")
    print(f"{name}: shape={df.shape}")
    print("First 15 columns:", actual[:15])

    expected_counts = Counter(expected)
    actual_counts = Counter(actual)

    missing = expected_counts - actual_counts
    extras = actual_counts - expected_counts

    print("Missing required columns:", dict(missing))
    print("Extra columns:", dict(extras))

    # Compare the ordered feature sequence using positional indices.
    # This handles duplicate names such as the two 'step' columns.
    positions = []
    used = set()

    for col in expected:
        match = next(
            (i for i, actual_col in enumerate(actual)
             if actual_col == col and i not in used),
            None
        )
        if match is None:
            positions = []
            break
        positions.append(match)
        used.add(match)

    if positions:
        selected_names = [actual[i] for i in positions]
        print("Can select all 82 features:", len(positions) == 82)
        print("Exact expected order:", selected_names == expected)
        print("Feature positions:", positions[:15], "...")
    else:
        print("Could not map every expected feature.")


In [0]:

# FACTORYMIND — ANOMALY INFERENCE
import numpy as np
import pandas as pd
from collections import Counter

expected = list(iso_forest.feature_names_in_)

# Select columns positionally so duplicate "step" names are preserved.
actual_columns = list(pdf.columns)
used = set()
feature_positions = []

for col in expected:
    match = next(
        (i for i, actual_col in enumerate(actual_columns)
         if actual_col == col and i not in used),
        None
    )
    if match is None:
        raise ValueError(f"Missing required anomaly feature: {col}")
    feature_positions.append(match)
    used.add(match)

X_anomaly = pdf.iloc[:, feature_positions].copy()

assert X_anomaly.shape[1] == iso_forest.n_features_in_
assert list(X_anomaly.columns) == expected

# Match the model's original feature order.
# Handle any missing values before inference.
X_anomaly = X_anomaly.replace([np.inf, -np.inf], np.nan)

valid_mask = X_anomaly.notna().all(axis=1)
X_anomaly_valid = X_anomaly.loc[valid_mask]

anomaly_predictions = iso_forest.predict(X_anomaly_valid)
anomaly_scores = iso_forest.decision_function(X_anomaly_valid)

# Build results using metadata from the same source rows.
machine_health_anomaly = pdf.loc[
    X_anomaly_valid.index, ["machine_id"]
].copy()

# Use the first "step" column for the metadata key.
step_position = actual_columns.index("step")
machine_health_anomaly["step"] = pdf.iloc[
    X_anomaly_valid.index, step_position
].to_numpy()

machine_health_anomaly["anomaly_prediction"] = anomaly_predictions
machine_health_anomaly["anomaly_score"] = anomaly_scores
machine_health_anomaly["is_anomaly"] = (
    machine_health_anomaly["anomaly_prediction"] == -1
)

print("Input rows:", len(pdf))
print("Valid rows scored:", len(machine_health_anomaly))
print("Rows skipped due to missing/non-finite features:",
      len(pdf) - len(machine_health_anomaly))
print("Anomaly rate:",
      round(machine_health_anomaly["is_anomaly"].mean() * 100, 2), "%")

display(machine_health_anomaly.head(10))


In [0]:

# FACTORYMIND — COMBINE ANOMALY DETECTION + RUL

# 1. Check uniqueness of join keys
anomaly_duplicates = machine_health_anomaly.duplicated(
    ["machine_id", "step"]
).sum()

rul_duplicates = machine_predictions.duplicated(
    ["machine_id", "step"]
).sum()

print("Duplicate anomaly keys:", anomaly_duplicates)
print("Duplicate RUL keys:", rul_duplicates)

if anomaly_duplicates > 0 or rul_duplicates > 0:
    raise ValueError(
        "Duplicate machine_id/step keys found. Resolve before joining."
    )

# 2. Select the RUL output fields
rul_cols = [
    "machine_id",
    "step",
    "predicted_rul",
    "model_version"
]
rul_for_join = machine_predictions[rul_cols].copy()

# 3. Join by machine and timestep
machine_health_assessment = machine_health_anomaly.merge(
    rul_for_join,
    on=["machine_id", "step"],
    how="inner",
    validate="one_to_one",
    indicator=True
)

# 4. Summarize join coverage
print("\nAnomaly rows:", len(machine_health_anomaly))
print("RUL rows:", len(rul_for_join))
print("Matched health rows:", len(machine_health_assessment))
print(
    "Anomaly rows without RUL match:",
    len(machine_health_anomaly) - len(machine_health_assessment)
)

# 5. Add a transparent initial health status
machine_health_assessment["health_status"] = np.select(
    [
        machine_health_assessment["is_anomaly"]
        & (machine_health_assessment["predicted_rul"] <= 120),

        machine_health_assessment["is_anomaly"],

        machine_health_assessment["predicted_rul"] <= 120
    ],
    [
        "CRITICAL",
        "ANOMALY_DETECTED",
        "RUL_WARNING"
    ],
    default="NORMAL"
)

display(
    machine_health_assessment.sort_values(
        ["machine_id", "step"]
    ).head(20)
)


In [0]:

# FACTORYMIND — DIAGNOSE RUL/ANOMALY JOIN COVERAGE

anomaly_keys = machine_health_anomaly[["machine_id", "step"]].drop_duplicates()
rul_keys = machine_predictions[["machine_id", "step"]].drop_duplicates()

unmatched_rul = rul_keys.merge(
    anomaly_keys,
    on=["machine_id", "step"],
    how="left",
    indicator=True,
    validate="one_to_one"
)

unmatched_rul = unmatched_rul[
    unmatched_rul["_merge"] == "left_only"
].drop(columns="_merge")

print("Unmatched RUL keys:", len(unmatched_rul))
display(unmatched_rul.head(50))

# Check whether the affected machines exist in the anomaly dataset.
if len(unmatched_rul):
    anomaly_machine_ids = set(machine_health_anomaly["machine_id"].unique())
    unmatched_rul["machine_exists_in_anomaly_data"] = (
        unmatched_rul["machine_id"].isin(anomaly_machine_ids)
    )
    display(unmatched_rul.head(50))


In [0]:

# FACTORYMIND — DIAGNOSE EARLY-STEP FEATURE AVAILABILITY
import numpy as np
import pandas as pd

actual_columns = list(pdf.columns)
expected = list(iso_forest.feature_names_in_)

# Map expected feature names to unique column positions.
used = set()
feature_positions = []

for col in expected:
    match = next(
        (
            i for i, name in enumerate(actual_columns)
            if name == col and i not in used
        ),
        None
    )
    if match is None:
        raise ValueError(f"Missing anomaly feature: {col}")
    feature_positions.append(match)
    used.add(match)

metadata_step_pos = actual_columns.index("step")

for machine_id in sorted(unmatched_rul["machine_id"].unique()):
    print(f"\n--- {machine_id} ---")

    rows = pdf.loc[pdf["machine_id"].eq(machine_id)].copy()

    # Sort positionally to avoid duplicate-column-name errors.
    order = np.argsort(
        pd.to_numeric(
            rows.iloc[:, metadata_step_pos], errors="coerce"
        ).to_numpy()
    )
    rows = rows.iloc[order].head(8)

    X_rows = rows.iloc[:, feature_positions].copy()

    # Convert each feature independently to numeric.
    X_numeric = X_rows.apply(pd.to_numeric, errors="coerce")
    values = X_numeric.to_numpy(dtype=float)

    diagnostic = pd.DataFrame({
        "machine_id": rows["machine_id"].to_numpy(),
        "step": rows.iloc[:, metadata_step_pos].to_numpy(),
        "missing_or_non_numeric": np.isnan(values).sum(axis=1),
        "infinite_values": np.isinf(values).sum(axis=1)
    })

    display(diagnostic)


In [0]:

# FACTORYMIND — COMBINED MACHINE HEALTH ASSESSMENT
import numpy as np
import pandas as pd

# 1. Prepare unique anomaly results
anomaly_results = machine_health_anomaly[
    ["machine_id", "step", "anomaly_score", "is_anomaly"]
].copy()

rul_results = machine_predictions[
    [
        "machine_id",
        "step",
        "failure_mode",
        "actual_rul",
        "predicted_rul",
        "prediction_error",
        "absolute_error",
        "model_version",
    ]
].copy()

# 2. Preserve every RUL evaluation row
machine_health_assessment = rul_results.merge(
    anomaly_results,
    on=["machine_id", "step"],
    how="left",
    validate="one_to_one",
    indicator=True,
)

machine_health_assessment["anomaly_status"] = np.where(
    machine_health_assessment["_merge"].eq("both"),
    np.where(
        machine_health_assessment["is_anomaly"].eq(True),
        "ANOMALY_DETECTED",
        "NORMAL",
    ),
    "UNAVAILABLE",
)

# 3. Apply a transparent preliminary status rule.
# Assumes each simulation step represents one minute.
rul_warning_threshold = 120

machine_health_assessment["health_status"] = np.select(
    [
        machine_health_assessment["anomaly_status"].eq("UNAVAILABLE"),
        machine_health_assessment["is_anomaly"].eq(True)
        & (machine_health_assessment["predicted_rul"] <= rul_warning_threshold),
        machine_health_assessment["is_anomaly"].eq(True),
        machine_health_assessment["predicted_rul"] <= rul_warning_threshold,
    ],
    [
        "ANOMALY_STATUS_UNAVAILABLE",
        "CRITICAL",
        "ANOMALY_DETECTED",
        "RUL_WARNING",
    ],
    default="NORMAL",
)

machine_health_assessment = machine_health_assessment.drop(
    columns=["_merge"]
)

# 4. Validate coverage
print("RUL evaluation rows:", len(rul_results))
print("Combined assessment rows:", len(machine_health_assessment))
print(
    "Rows without anomaly inference:",
    int(
        machine_health_assessment["anomaly_status"]
        .eq("UNAVAILABLE").sum()
    ),
)

print("\nAnomaly status counts:")
display(
    machine_health_assessment["anomaly_status"]
    .value_counts(dropna=False)
    .rename_axis("anomaly_status")
    .reset_index(name="row_count")
)

print("\nCombined health status counts:")
display(
    machine_health_assessment["health_status"]
    .value_counts(dropna=False)
    .rename_axis("health_status")
    .reset_index(name="row_count")
)

print("\nSample assessments:")
display(
    machine_health_assessment.sort_values(
        ["machine_id", "step"]
    ).head(20)
)

assert len(machine_health_assessment) == len(rul_results)
assert not machine_health_assessment.duplicated(
    ["machine_id", "step"]
).any()


In [0]:

# FACTORYMIND — RCA EVIDENCE EXTRACTION
import numpy as np
import pandas as pd

# Use the existing engineered telemetry dataset.
# pdf contains duplicate column names, so select sensor columns by name
# only where their labels are unique.
sensor_names = [
    "temperature",
    "vibration",
    "pressure",
    "motor_current",
    "rpm",
    "load",
    "coolant_temperature",
    "coolant_flow",
    "acoustic_level",
]

columns = list(pdf.columns)

def unique_column_position(name):
    positions = [i for i, col in enumerate(columns) if col == name]
    if len(positions) != 1:
        raise ValueError(
            f"Expected one '{name}' column in pdf, found {len(positions)}"
        )
    return positions[0]

# Extract machine/step metadata and sensor readings safely.
metadata = pd.DataFrame({
    "machine_id": pdf["machine_id"].to_numpy(),
    "step": pdf.iloc[:, columns.index("step")].to_numpy(),
})

sensor_data = pd.DataFrame({
    sensor: pd.to_numeric(
        pdf.iloc[:, unique_column_position(sensor)], errors="coerce"
    ).to_numpy()
    for sensor in sensor_names
})

# Build a machine-specific rolling baseline.
# Current values are compared with the previous 30 steps,
# avoiding inclusion of the current reading in its own baseline.
ordered = pd.concat([metadata, sensor_data], axis=1)
ordered["_original_order"] = np.arange(len(ordered))
ordered = ordered.sort_values(
    ["machine_id", "step", "_original_order"]
).reset_index(drop=True)

rca_parts = []

for machine_id, group in ordered.groupby("machine_id", sort=False):
    group = group.copy()

    for sensor in sensor_names:
        history = group[sensor].shift(1)
        rolling_mean = history.rolling(30, min_periods=10).mean()
        rolling_std = history.rolling(30, min_periods=10).std()

        group[f"{sensor}_baseline"] = rolling_mean
        group[f"{sensor}_zscore"] = (
            (group[sensor] - rolling_mean)
            / rolling_std.replace(0, np.nan)
        )

    rca_parts.append(group)

rca_evidence = pd.concat(rca_parts, ignore_index=True)

# Join RCA evidence to the combined health assessment.
zscore_columns = [f"{s}_zscore" for s in sensor_names]

rca_join = machine_health_assessment.merge(
    rca_evidence[
        ["machine_id", "step"] + zscore_columns
    ],
    on=["machine_id", "step"],
    how="left",
    validate="one_to_one",
)

# Choose the sensor with the largest absolute standardized deviation.
z_matrix = rca_join[zscore_columns].to_numpy(dtype=float)
valid_counts = np.isfinite(z_matrix).sum(axis=1)
safe_z = np.where(np.isfinite(z_matrix), z_matrix, np.nan)

root_sensor = []
root_zscore = []

for row in safe_z:
    if not np.isfinite(row).any():
        root_sensor.append("INSUFFICIENT_HISTORY")
        root_zscore.append(np.nan)
    else:
        idx = int(np.nanargmax(np.abs(row)))
        root_sensor.append(sensor_names[idx])
        root_zscore.append(float(row[idx]))

rca_join["leading_sensor_deviation"] = root_sensor
rca_join["leading_sensor_zscore"] = root_zscore
rca_join["rca_evidence_status"] = np.where(
    valid_counts > 0,
    "SENSOR_DEVIATION_AVAILABLE",
    "INSUFFICIENT_HISTORY",
)

print("RCA assessment rows:", len(rca_join))
print("\nRCA evidence status:")
display(
    rca_join["rca_evidence_status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="rows")
)

print("\nSample RCA results:")
display(
    rca_join[
        [
            "machine_id",
            "step",
            "failure_mode",
            "health_status",
            "leading_sensor_deviation",
            "leading_sensor_zscore",
            "rca_evidence_status",
        ]
    ].head(20)
)

assert len(rca_join) == len(machine_health_assessment)


In [0]:

# FACTORYMIND — DETERMINISTIC POLICY DECISIONS
import pandas as pd
import numpy as np

policy_input = rca_join.copy()

# Explicit policy thresholds for this simulation.
RUL_WARNING_MINUTES = 120
RUL_CRITICAL_MINUTES = 30
SENSOR_DEVIATION_THRESHOLD = 3.0

def evaluate_factorymind_policy(row):
    rul = row.get("predicted_rul")
    anomaly_status = row.get("anomaly_status", "UNAVAILABLE")
    sensor_z = row.get("leading_sensor_zscore")
    sensor = row.get("leading_sensor_deviation", "UNKNOWN")

    # Missing anomaly evidence must never be treated as normal.
    if pd.isna(rul) or anomaly_status == "UNAVAILABLE":
        return pd.Series({
            "policy_decision": "REVIEW_REQUIRED",
            "recommended_action": "Collect missing evidence and review",
            "decision_reason": "Incomplete anomaly or RUL evidence",
            "requires_human_approval": True,
        })

    if rul <= RUL_CRITICAL_MINUTES:
        return pd.Series({
            "policy_decision": "CRITICAL",
            "recommended_action": "Escalate for urgent maintenance review",
            "decision_reason": f"Predicted RUL <= {RUL_CRITICAL_MINUTES} minutes",
            "requires_human_approval": True,
        })

    if anomaly_status == "ANOMALY_DETECTED":
        if pd.notna(sensor_z) and abs(sensor_z) >= SENSOR_DEVIATION_THRESHOLD:
            return pd.Series({
                "policy_decision": "INVESTIGATE_ANOMALY",
                "recommended_action": f"Inspect {sensor} and verify sensor readings",
                "decision_reason": (
                    f"Anomaly detected; leading sensor deviation "
                    f"|z|={abs(sensor_z):.2f}"
                ),
                "requires_human_approval": True,
            })

        return pd.Series({
            "policy_decision": "INVESTIGATE_ANOMALY",
            "recommended_action": "Inspect telemetry and review machine condition",
            "decision_reason": "Anomaly detected by Isolation Forest",
            "requires_human_approval": True,
        })

    if rul <= RUL_WARNING_MINUTES:
        return pd.Series({
            "policy_decision": "RUL_WARNING",
            "recommended_action": "Schedule maintenance assessment",
            "decision_reason": f"Predicted RUL <= {RUL_WARNING_MINUTES} minutes",
            "requires_human_approval": True,
        })

    return pd.Series({
        "policy_decision": "MONITOR",
        "recommended_action": "Continue monitoring; no automated intervention",
        "decision_reason": "No current policy trigger",
        "requires_human_approval": False,
    })

policy_output = policy_input.join(
    policy_input.apply(evaluate_factorymind_policy, axis=1)
)

print("Policy rows:", len(policy_output))
print("\nPolicy decision counts:")
display(
    policy_output["policy_decision"]
    .value_counts()
    .rename_axis("policy_decision")
    .reset_index(name="row_count")
)

print("\nSample policy decisions:")
display(
    policy_output[
        [
            "machine_id",
            "step",
            "failure_mode",
            "predicted_rul",
            "anomaly_status",
            "leading_sensor_deviation",
            "policy_decision",
            "recommended_action",
            "requires_human_approval",
        ]
    ].head(20)
)

assert len(policy_output) == len(rca_join)
assert not policy_output.duplicated(["machine_id", "step"]).any()


In [0]:

# FACTORYMIND — DISCOVER EXISTING POLICY / TOOLBROKER / AUDIT APIs
import inspect
import sys

keywords = (
    "policy", "broker", "audit", "approval",
    "rca", "maintenance", "tool"
)

print("=== Relevant loaded classes and functions ===")

for name, obj in list(globals().items()):
    if not any(k in name.lower() for k in keywords):
        continue

    if inspect.isclass(obj) or inspect.isfunction(obj):
        try:
            print(f"\n{name}: {obj.__module__}")
            print("Signature:", inspect.signature(obj))
            print("Source file:", inspect.getfile(obj))
        except (TypeError, ValueError, OSError):
            print(f"\n{name}: signature/source unavailable")

print("\n=== Relevant loaded project modules ===")

for module_name, module in sorted(sys.modules.items()):
    if (
        module is not None
        and any(k in module_name.lower() for k in keywords)
        and "factorymind" in module_name.lower()
    ):
        print(module_name, getattr(module, "__file__", None))


In [0]:

# FACTORYMIND — LOCATE EXISTING CORE IMPLEMENTATION
import os

project_root = "/Workspace/Users/praveenbabugeddada7@gmail.com/factorymind"

print("Project directory exists:", os.path.isdir(project_root))

if os.path.isdir(project_root):
    print("\nTop-level entries:")
    for item in sorted(os.listdir(project_root)):
        full_path = os.path.join(project_root, item)
        print(("DIR  " if os.path.isdir(full_path) else "FILE "), item)

    print("\nRelevant Python files:")
    keywords = (
        "policy", "broker", "audit", "approval",
        "rca", "tool", "maintenance", "workflow"
    )

    for root, dirs, files in os.walk(project_root):
        # Avoid scanning environments and generated artifacts.
        dirs[:] = [
            d for d in dirs
            if d not in {
                ".git", "__pycache__", ".venv", "venv",
                "node_modules", "artifacts"
            }
        ]

        for filename in sorted(files):
            if filename.endswith(".py") and any(
                word in filename.lower() for word in keywords
            ):
                print(os.path.join(root, filename))
else:
    print(
        "Workspace path is not accessible through the local filesystem. "
        "We will locate the source through the Databricks Workspace interface."
    )


In [0]:

# FACTORYMIND — RESOLVE ACTUAL SOURCE PATHS
from pathlib import Path

ROOT = Path(
    "/Workspace/Users/praveenbabugeddada@gmail.com/factorymind"
)

print("Project root:", ROOT)
print("Resolved root:", ROOT.resolve())
print("Root exists:", ROOT.exists())

targets = {
    "broker.py",
    "machine_tools.py",
    "test_policy.py",
    "test_rca.py",
    "test_audit.py",
    "test_tools.py",
}

for path in ROOT.rglob("*.py"):
    if path.name in targets:
        print(f"\n{'=' * 80}")
        print("FILE:", path)
        print("=" * 80)

        try:
            print(path.read_text(encoding="utf-8")[:12000])
        except Exception as exc:
            print("Could not read file:", repr(exc))


In [0]:

# FACTORYMIND — CHECK SOURCE DIRECTORY ACCESS
import os

ROOT = "/Workspace/Users/praveenbabugeddada@gmail.com/factorymind"

paths = [
    ROOT,
    os.path.join(ROOT, "factorymind"),
    os.path.join(ROOT, "factorymind", "tools"),
    os.path.join(ROOT, "tests"),
    os.path.join(ROOT, "tests", "unit"),
]

for path in paths:
    print(f"\nPATH: {path}")
    print("Exists:", os.path.exists(path))
    print("Directory:", os.path.isdir(path))

    if os.path.isdir(path):
        try:
            entries = os.listdir(path)
            print("Entries:", entries[:40])
        except Exception as exc:
            print("Listing error:", repr(exc))

# Check direct access to the expected source files.
for relative in [
    "factorymind/tools/broker.py",
    "factorymind/tools/machine_tools.py",
    "tests/unit/test_policy.py",
    "tests/unit/test_rca.py",
    "tests/unit/test_audit.py",
]:
    path = os.path.join(ROOT, relative)
    print(f"\n{relative}")
    print("Exists:", os.path.exists(path))
    print("Regular file:", os.path.isfile(path))


In [0]:

# FACTORYMIND — CHECK PACKAGE IMPORTS
import importlib.util
import os
import sys

print("Python executable:", sys.executable)
print("Current directory:", os.getcwd())

for name in [
    "factorymind",
    "factorymind.tools",
    "factorymind.tools.broker",
    "factorymind.tools.machine_tools",
]:
    try:
        spec = importlib.util.find_spec(name)
        print(f"{name}: {spec.origin if spec else 'NOT FOUND'}")
    except Exception as exc:
        print(f"{name}: ERROR — {exc}")


In [0]:

# FACTORYMIND — INSPECT EXISTING CORE APIs
import importlib
import inspect
import pkgutil
import factorymind

# Discover the actual module names without guessing their filenames.
print("=== FactoryMind modules ===")
modules = sorted(
    item.name
    for item in pkgutil.walk_packages(
        factorymind.__path__,
        prefix="factorymind."
    )
)

for name in modules:
    if any(k in name.lower() for k in [
        "policy", "rca", "audit", "broker",
        "tool", "approval"
    ]):
        print(name)

# Inspect the known ToolBroker and machine-tool modules.
for module_name in [
    "factorymind.tools.broker",
    "factorymind.tools.machine_tools",
]:
    print(f"\n{'=' * 75}")
    print("MODULE:", module_name)
    print("=" * 75)

    module = importlib.import_module(module_name)

    for name, obj in inspect.getmembers(module):
        if name.startswith("_"):
            continue

        if inspect.isclass(obj) or inspect.isfunction(obj):
            # Exclude objects imported from unrelated modules.
            if getattr(obj, "__module__", None) != module.__name__:
                continue

            try:
                signature = inspect.signature(obj)
            except (TypeError, ValueError):
                signature = "(signature unavailable)"

            print(f"\n{name}{signature}")

            if inspect.isclass(obj):
                for method_name, method in inspect.getmembers(
                    obj, predicate=inspect.isfunction
                ):
                    if not method_name.startswith("_"):
                        try:
                            print(
                                f"  .{method_name}"
                                f"{inspect.signature(method)}"
                            )
                        except (TypeError, ValueError):
                            print(f"  .{method_name}(...)")


In [0]:

# FACTORYMIND — INSPECT POLICY, RCA, REGISTRY AND AUDIT CONTRACTS
import importlib
import inspect

modules_to_inspect = [
    "factorymind.policy.engine",
    "factorymind.policy.rules",
    "factorymind.policy.permissions",
    "factorymind.rca.engine",
    "factorymind.rca.evidence",
    "factorymind.rca.signatures",
    "factorymind.tools.registry",
    "factorymind.tools.schemas",
    "factorymind.audit.chain",
    "factorymind.audit.events",
    "factorymind.audit.verification",
]

for module_name in modules_to_inspect:
    module = importlib.import_module(module_name)

    print("\n" + "=" * 80)
    print("MODULE:", module_name)
    print("=" * 80)

    for name, obj in inspect.getmembers(module):
        if name.startswith("_"):
            continue

        if not (inspect.isclass(obj) or inspect.isfunction(obj)):
            continue

        if getattr(obj, "__module__", None) != module.__name__:
            continue

        try:
            signature = inspect.signature(obj)
        except (TypeError, ValueError):
            signature = "(signature unavailable)"

        print(f"\n{name}{signature}")

        if inspect.isclass(obj):
            for method_name, method in inspect.getmembers(
                obj, predicate=inspect.isfunction
            ):
                if method_name.startswith("_"):
                    continue

                try:
                    method_signature = inspect.signature(method)
                except (TypeError, ValueError):
                    method_signature = "(signature unavailable)"

                print(f"  .{method_name}{method_signature}")


In [0]:

# FACTORYMIND — INITIALIZE CORE COMPONENTS
from factorymind.policy.engine import PolicyEngine
from factorymind.tools.registry import ToolRegistry
from factorymind.tools.broker import ToolBroker
from factorymind.audit.chain import AuditChain
from factorymind.audit.verification import ChainVerifier
from factorymind.rca.engine import RCAEngine

policy_engine_core = PolicyEngine()
tool_registry_core = ToolRegistry()
audit_chain_core = AuditChain()
rca_engine_core = RCAEngine()

# Inspect registered tool names and policy enum values.
print("=== Registered tools ===")
print(tool_registry_core.list_tool_names())

print("\n=== Relevant enum definitions ===")
import inspect
from factorymind.domain import enums

for name in ["ToolName", "HealthState", "MachineState", "FailureMode"]:
    enum_cls = getattr(enums, name, None)
    print(f"\n{name}:")
    if enum_cls is not None:
        try:
            print(list(enum_cls))
        except TypeError:
            print(inspect.getsource(enum_cls))
    else:
        print("Not found in factorymind.domain.enums")

# Create the broker using the real implementations.
tool_broker_core = ToolBroker(
    registry=tool_registry_core,
    policy_engine=policy_engine_core,
    audit_chain=audit_chain_core,
)

print("\nCore components initialized.")
print("Audit events initially:", len(audit_chain_core.export()))


In [0]:

# FACTORYMIND — STANDALONE TOOLBROKER SMOKE TEST
from factorymind.domain.enums import (
    ToolName,
    HealthState,
    MachineState,
    FailureMode,
)
from factorymind.tools.broker import ToolBroker
from factorymind.tools.registry import ToolRegistry
from factorymind.policy.engine import PolicyEngine
from factorymind.audit.chain import AuditChain
from factorymind.audit.verification import ChainVerifier

# Create fresh instances so the test is independent of notebook state.
policy_engine_core = PolicyEngine()
tool_registry_core = ToolRegistry()
audit_chain_core = AuditChain()

tool_broker_core = ToolBroker(
    registry=tool_registry_core,
    policy_engine=policy_engine_core,
    audit_chain=audit_chain_core,
)

print("Registered tools:", tool_registry_core.list_tool_names())

# Confirm that the read-only status tool is registered.
assert tool_registry_core.exists(ToolName.GET_MACHINE_STATUS), (
    "get_machine_status is not registered in this ToolRegistry."
)

# Execute a read-only request with explicit test context.
broker_result = tool_broker_core.request_execution(
    tool_name=ToolName.GET_MACHINE_STATUS,
    machine_id="M-0000",
    params={},
    actor="factorymind_databricks_test",
    machine_state=MachineState.RUNNING,
    health_state=HealthState.HEALTHY,
    degradation_score=0.0,
    root_cause=FailureMode.NONE,
)

print("\n=== Broker result ===")
print(broker_result)

print("\n=== Audit events ===")
events = audit_chain_core.export()
print("Event count:", len(events))
for event in events:
    print(event)

print("\n=== Audit verification ===")
verification_result = ChainVerifier().verify(audit_chain_core)
print(verification_result)


In [0]:

# FACTORYMIND — INSPECT DOMAIN MODELS AND POLICY RESULT TYPES
import inspect

from factorymind.domain import models, enums
from factorymind.policy import engine, rules

for module, names in [
    (models, ["MachineConfig", "TelemetrySnapshot", "MachineState"]),
    (enums, ["HealthState", "MachineState", "FailureMode", "ToolName"]),
    (rules, ["PolicyContext"]),
]:
    print(f"\n=== {module.__name__} ===")
    for name in names:
        obj = getattr(module, name, None)
        if obj is None:
            continue
        try:
            print(f"\n{name}{inspect.signature(obj)}")
            if inspect.isclass(obj):
                print("Fields:", getattr(obj, "__annotations__", {}))
        except (TypeError, ValueError):
            print(name, obj)

print("\n=== Policy decision result ===")
policy_module = inspect.getmodule(engine.PolicyEngine.evaluate)
for name, obj in inspect.getmembers(policy_module, inspect.isclass):
    if "Decision" in name or "Policy" in name:
        if getattr(obj, "__module__", None) == policy_module.__name__:
            try:
                print(name, inspect.signature(obj))
                print("Fields:", getattr(obj, "__annotations__", {}))
            except (TypeError, ValueError):
                pass


In [0]:

# FACTORYMIND — INSPECT RCA / POLICY RESULT CONTRACTS
import inspect
import dataclasses

from factorymind.rca.engine import RCAEngine
from factorymind.policy.engine import PolicyEngine
from factorymind.policy import rules

# Find the result classes referenced by the APIs.
for module in [rules, inspect.getmodule(RCAEngine.analyze)]:
    print(f"\n=== {module.__name__} ===")

    for name, obj in inspect.getmembers(module, inspect.isclass):
        if getattr(obj, "__module__", None) != module.__name__:
            continue

        if any(word in name.lower() for word in [
            "decision", "result", "cause", "evidence", "anomaly"
        ]):
            print(f"\n{name}")

            if dataclasses.is_dataclass(obj):
                print("Fields:", [
                    f.name for f in dataclasses.fields(obj)
                ])
            else:
                print("Annotations:", getattr(obj, "__annotations__", {}))

            try:
                print("Constructor:", inspect.signature(obj))
            except (TypeError, ValueError):
                pass

# Also inspect the RCA analyze implementation to see how results are built.
print("\n=== RCAEngine.analyze source ===")
print(inspect.getsource(RCAEngine.analyze))

print("\n=== PolicyEngine.evaluate source ===")
print(inspect.getsource(PolicyEngine.evaluate))


In [0]:

# FACTORYMIND — POLICY PRECEDENCE CHECK
import inspect

from factorymind.policy import engine as policy_engine_module
from factorymind.policy import rules as policy_rules_module

from factorymind.domain.enums import (
    HealthState,
    MachineState,
    FailureMode,
    ToolName,
)

# Show the actual ordered rules used by PolicyEngine.evaluate.
policy_rules = getattr(policy_engine_module, "POLICY_RULES", None)

if policy_rules is None:
    policy_rules = getattr(policy_rules_module, "POLICY_RULES", None)

if policy_rules is None:
    print("POLICY_RULES not found in the expected modules.")
else:
    print("=== Policy rule precedence ===")
    for index, rule in enumerate(policy_rules, start=1):
        print(f"{index}. {rule.__name__}")

# Find PolicyDecision and its supported values.
PolicyDecision = getattr(policy_rules_module, "PolicyDecision", None)

if PolicyDecision is not None:
    print("\n=== PolicyDecision values ===")
    try:
        print(list(PolicyDecision))
    except TypeError:
        print("Members:", getattr(PolicyDecision, "__members__", {}))

print("\n=== Domain states ===")
print("HealthState:", list(HealthState))
print("MachineState:", list(MachineState))
print("FailureMode:", list(FailureMode))
print("ToolName:", list(ToolName))


In [0]:

import inspect
import importlib

from factorymind.policy.engine import PolicyEngine
from factorymind.policy import engine as policy_engine_module

# Find where PolicyContext is actually defined or imported.
PolicyContext = getattr(policy_engine_module, "PolicyContext", None)

if PolicyContext is not None:
    print("PolicyContext module:", PolicyContext.__module__)
    print("Constructor:", inspect.signature(PolicyContext))
else:
    print("PolicyContext is not exposed by policy.engine.")

    # Inspect likely package modules without reading Workspace files.
    for module_name in [
        "factorymind.policy",
        "factorymind.policy.engine",
        "factorymind.policy.rules",
        "factorymind.policy.permissions",
    ]:
        try:
            module = importlib.import_module(module_name)
            matches = [
                name for name, obj in vars(module).items()
                if name == "PolicyContext"
                or (
                    inspect.isclass(obj)
                    and "PolicyContext" in name
                )
            ]
            print(module_name, "->", matches)
        except Exception as exc:
            print(module_name, "->", type(exc).__name__, str(exc))

# Show the exact imports used by the policy engine.
print("\n=== Policy engine imports ===")
print(inspect.getsource(PolicyEngine.evaluate))


In [0]:

from factorymind.tools.registry import ToolRegistry
from factorymind.domain.enums import ToolName
import inspect

registry = ToolRegistry()
tool_name = ToolName.REDUCE_MACHINE_LOAD

print("=== Tool definition ===")
tool = registry.get(tool_name)
print(tool)

print("\n=== Tool definition structure ===")
print("Type:", type(tool))
print("Attributes:", [
    name for name in dir(tool)
    if not name.startswith("_")
])

print("\n=== Registered tool names ===")
print(registry.list_tool_names())

print("\n=== Broker execution method ===")
from factorymind.tools.broker import ToolBroker
print(inspect.signature(ToolBroker.request_execution))


In [0]:

import inspect
from factorymind.tools.broker import ToolBroker

print("=== ToolBroker.request_execution implementation ===")
print(inspect.getsource(ToolBroker.request_execution))
